# 05 — Explain Mechanism

EXPLAIN — the mechanistic backbone (theory-anchored on Minority Collapse). Hypothesis tournament + neural-collapse geometry + per-family proximate causes. The surviving mechanism DEFINES the diagnostic's feature set.

In [ ]:
# ── FULL bootstrap — run FIRST ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch, pandas as pd, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU",
      "| split:", CFG['split']['primary'])

Mounted at /content/drive
GPU: Tesla T4 | split: temporal_within_capture


In [ ]:
# --- Colab bootstrap (config-driven; never hardcode a path) ---
try:
    from google.colab import drive; drive.mount('/content/drive')
    REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
except Exception:
    REPO = '.'
import os, sys
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds, require_frozen
set_all_seeds(CFG['anchor_seed'])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from src import data
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("rows:", len(df), "| test:", len(splits["test"]))

rows: 3661696 | test: 549271


In [ ]:
import sys
for m in list(sys.modules):
    if m.startswith("src"): del sys.modules[m]   # clear stale cache

try:
    import src.explain
    print("explain imported OK")
except Exception as e:
    import traceback; traceback.print_exc()

Traceback (most recent call last):
  File "/tmp/ipykernel_10661/4097806076.py", line 6, in <cell line: 0>
    import src.explain
ModuleNotFoundError: No module named 'src.explain'


In [ ]:
import os
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
print("explain.py exists:", os.path.exists(f"{REPO}/src/explain.py"))
print("\nfiles in src/:")
for f in sorted(os.listdir(f"{REPO}/src")):
    p = f"{REPO}/src/{f}"
    if os.path.isfile(p):
        print(f"  {f}  ({os.path.getsize(p)} bytes)")
print("\nimporting src from:", os.path.dirname(__import__('src').__file__))

explain.py exists: True

files in src/:
  __init__.py  (0 bytes)
  compression.py  (10934 bytes)
  config.py  (4438 bytes)
  crux.py  (1161 bytes)
  data.py  (10577 bytes)
  diagnostic.py  (1892 bytes)
  explain.py  (5578 bytes)
  geometry.py  (4141 bytes)
  metrics.py  (4458 bytes)
  mitigate.py  (1639 bytes)
  models.py  (3897 bytes)
  train.py  (7539 bytes)

importing src from: /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src


In [ ]:
import importlib.util, sys
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
spec = importlib.util.spec_from_file_location("src.explain", f"{REPO}/src/explain.py")
explain = importlib.util.module_from_spec(spec)
sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
print("explain loaded:", hasattr(explain, "per_class_geometry"))

explain loaded: True


In [ ]:
import importlib
from src import data, train, compression, models
for m in (data, train, compression): importlib.reload(m)
import pandas as pd, numpy as np, torch

# 1) reload M0 anchor, rebuild prune80
anchor, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",
                                                  CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
prune80, _le, _sc = compression.prune_and_finetune(anchor, df, "ciciot2023", splits,
                                                   CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
print("M0 + prune80 ready")

# 2) extract penultimate features from BOTH models
fM0, lM0, y = explain.extract_features(anchor, df, splits, scaler, feat_cols, le)
fP8, lP8, _ = explain.extract_features(prune80, df, splits, scaler, feat_cols, le)
print("features:", fM0.shape, "->", fP8.shape)

# 3) per-class geometry on M0 + prune80 with bootstrap CIs
geomM0 = explain.per_class_geometry(fM0, lM0, y, le, n_boot=200, seed=0)
geomP8 = explain.per_class_geometry(fP8, lP8, y, le, n_boot=200, seed=0)

# 4) THE TEST: does baseline geometry predict prune80 Δrecall?
delta = pd.read_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"), index_col=0)["prune80"]
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]
meas = tiers[tiers=="measurable"].index

corr = explain.correlate_geometry_vs_collapse(geomM0.loc[meas], delta.loc[meas], "mean_cos_to_others")
print("\n=== HYPOTHESIS TEST: baseline geometry vs prune80 collapse (measurable tier) ===")
print({k:(round(v,4) if isinstance(v,float) else v) for k,v in corr.items()})

corr_m = explain.correlate_geometry_vs_collapse(geomM0.loc[meas], delta.loc[meas], "margin")
print("margin vs collapse:", {k:(round(v,4) if isinstance(v,float) else v) for k,v in corr_m.items()})

# 5) per-class geometry next to collapse, worst first
R = pd.read_csv(PATHS.tables("compression","cnn1d_per_class_recall_matrix.csv"),index_col=0)
tbl = pd.DataFrame({
    "M0_recall": R["M0"],
    "prune80_d_recall": delta,
    "M0_geom_cos": geomM0["mean_cos_to_others"],
    "M0_margin": geomM0["margin"],
    "prune80_geom_cos": geomP8["mean_cos_to_others"],
    "tier": tiers,
}).loc[meas].sort_values("prune80_d_recall")
print("\n=== per-class: baseline geometry vs collapse (worst first) ===")
print(tbl.round(3).to_string())

geomM0.to_csv(PATHS.tables("explain","cnn1d_M0_geometry.csv"))
geomP8.to_csv(PATHS.tables("explain","cnn1d_prune80_geometry.csv"))
tbl.to_csv(PATHS.tables("explain","geometry_vs_collapse.csv"))
print("\nsaved geometry tables -> results/tables/explain/")

M0 + prune80 ready
features: (549271, 128) -> (549271, 128)

=== HYPOTHESIS TEST: baseline geometry vs prune80 collapse (measurable tier) ===
{'n': 13, 'spearman_r': 0.1923, 'spearman_p': 0.5291, 'kendall_tau': 0.1538, 'kendall_p': 0.5098, 'interpretation': 'no/positive relationship'}
margin vs collapse: {'n': 13, 'spearman_r': -0.2637, 'spearman_p': 0.3839, 'kendall_tau': -0.1795, 'kendall_p': 0.4354, 'interpretation': 'fragile baseline geometry predicts collapse'}

=== per-class: baseline geometry vs collapse (worst first) ===
                      M0_recall  prune80_d_recall  M0_geom_cos  M0_margin  prune80_geom_cos        tier
label                                                                                                  
DoS-UDP_Flood             0.682            -0.682       -0.126      1.271            -0.182  measurable
DoS-HTTP_Flood            0.867            -0.654       -0.022      2.142             0.011  measurable
Recon-HostDiscovery       0.705            -0.629

In [ ]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
ADD = r'''

def confusability_reallocation(model_M0, model_comp, df, splits, scaler, feat_cols, le,
                               collapsed_classes, is_half_comp=False, which="test"):
    """Test the CONFUSABILITY mechanism: when a class collapses under compression,
    where do its samples go? For each collapsed class, find which OTHER class
    absorbs the most of its (now-misclassified) test samples, and whether that
    absorber's recall RISES. If collapse pairs with a sibling's gain, the
    mechanism is winner-take-all reallocation among confusable classes."""
    import numpy as np, pandas as pd
    fM0, lM0, y = extract_features(model_M0, df, splits, scaler, feat_cols, le, which=which)
    fC, lC, _ = extract_features(model_comp, df, splits, scaler, feat_cols, le,
                                 which=which, is_half=is_half_comp)
    predM0 = lM0.argmax(1); predC = lC.argmax(1)
    name = lambda i: le.classes_[int(i)]
    idx = {n: i for i, n in enumerate(le.classes_)}
    def recall(pred):
        return {int(c): float((pred[y == c] == c).mean()) if (y == c).any() else np.nan
                for c in np.unique(y)}
    rM0, rC = recall(predM0), recall(predC)
    rows = []
    for cls in collapsed_classes:
        c = idx[cls]
        mask = (y == c)
        comp_pred = predC[mask]
        wrong = comp_pred[comp_pred != c]
        if len(wrong) == 0:
            rows.append({"collapsed_class": cls, "top_absorber": None}); continue
        vals, counts = np.unique(wrong, return_counts=True)
        top = vals[counts.argmax()]
        frac = counts.max() / mask.sum()
        rows.append({
            "collapsed_class": cls,
            "M0_recall": round(rM0[c], 3),
            "comp_recall": round(rC[c], 3),
            "top_absorber": name(top),
            "absorber_took_frac": round(float(frac), 3),
            "absorber_recall_M0": round(rM0[int(top)], 3),
            "absorber_recall_comp": round(rC[int(top)], 3),
            "absorber_recall_gain": round(rC[int(top)] - rM0[int(top)], 3),
        })
    return pd.DataFrame(rows).sort_values("absorber_took_frac", ascending=False)
'''
with open(f"{REPO}/src/explain.py","a") as f:
    f.write(ADD)
print("appended confusability_reallocation")

appended confusability_reallocation


In [ ]:
# reload explain from file to pick up the new function (file-path bypass)
import importlib.util, sys
spec = importlib.util.spec_from_file_location("src.explain",
        "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src/explain.py")
explain = importlib.util.module_from_spec(spec); sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
import pandas as pd

# the 10 classes that collapsed under prune80 (from the collapse flags)
flags = pd.read_csv(PATHS.tables("compression","cnn1d_collapse_flags.csv"), index_col=0)
collapsed = list(flags.index[flags["prune80"]])
print("collapsed under prune80:", collapsed, "\n")

conf = explain.confusability_reallocation(anchor, prune80, df, splits, scaler, feat_cols, le, collapsed)
print("=== CONFUSABILITY: where did each collapsed class's samples go under prune80? ===")
print(conf.to_string(index=False))
conf.to_csv(PATHS.tables("explain","prune80_confusability.csv"), index=False)
print("\nsaved -> results/tables/explain/prune80_confusability.csv")

collapsed under prune80: ['BenignTraffic', 'BrowserHijacking', 'CommandInjection', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SlowLoris', 'DDoS-TCP_Flood', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-TCP_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Mirai-greeth_flood', 'Mirai-greip_flood', 'Mirai-udpplain', 'Recon-HostDiscovery', 'Recon-OSScan', 'Recon-PortScan', 'SqlInjection', 'XSS'] 

=== CONFUSABILITY: where did each collapsed class's samples go under prune80? ===
     collapsed_class  M0_recall  comp_recall            top_absorber  absorber_took_frac  absorber_recall_M0  absorber_recall_comp  absorber_recall_gain
       DoS-SYN_Flood      0.164        0.000          DDoS-SYN_Flood               0.973               0.523                 0.940                 0.417
       DoS-UDP_Flood      0.682        0.000          DDoS-UDP_Flood               0.956               0.742                 0.997                 0.255
    CommandInjection      0

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "EXPLAIN: simple geometry hypothesis NULL (Spearman +0.19/-0.26, ns); confusability test reveals two-regime collapse — protocol-confusable DoS/DDoS pairs consolidate onto sibling, AND 10 hard/sparse classes collapse into single absorber sink (VulnerabilityScan). Mechanism is structured sink-consolidation, not Minority Collapse"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   afc0add..c5f746f  main -> main



In [ ]:
import importlib
from src import data, train
importlib.reload(train)

# train the MLP anchor at the anchor seed (same recipe as CNN: tempered weights, early stop)
mlp, mlp_info = train.train_model(
    "mlp", df, "ciciot2023", splits, seed=CFG["anchor_seed"],
    epochs=40, patience=6, batch_size=4096, arch_kwargs={"hidden": (256, 128)})
print(f"\nMLP params={mlp_info['params']}  val_macroF1={mlp_info['macroF1_val']:.4f}  test_macroF1={mlp_info['macroF1_test']:.4f}")

# per-class recall at M0
yt, yp, probs = train.predict(mlp, df, splits, mlp_info["label_encoder"], mlp_info["scaler"], mlp_info["feat_cols"])
tab = train.per_class_recall_table(yt, yp, mlp_info["label_encoder"])
print("\nMLP M0 per-class recall:")
print(tab.to_string(index=False))
tab.to_csv(PATHS.tables("baseline","mlp_M0_per_class_recall.csv"), index=False)

  epoch 00  val_macroF1=0.5440
  epoch 01  val_macroF1=0.5386
  epoch 02  val_macroF1=0.5433
  epoch 03  val_macroF1=0.5593
  epoch 04  val_macroF1=0.5563
  epoch 05  val_macroF1=0.5595
  epoch 06  val_macroF1=0.5563
  epoch 07  val_macroF1=0.5355
  epoch 08  val_macroF1=0.5408
  epoch 09  val_macroF1=0.5559
  epoch 10  val_macroF1=0.5585
  epoch 11  val_macroF1=0.5638
  epoch 12  val_macroF1=0.5719
  epoch 13  val_macroF1=0.5715
  epoch 14  val_macroF1=0.5799
  epoch 15  val_macroF1=0.5767
  epoch 16  val_macroF1=0.5726
  epoch 17  val_macroF1=0.5748
  epoch 18  val_macroF1=0.5773
  epoch 19  val_macroF1=0.5739
  epoch 20  val_macroF1=0.5679
  early stop @ epoch 20
  saved -> /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/models/ciciot2023/ciciot2023__mlp__M0__seed0.pt

MLP params=48290  val_macroF1=0.5799  test_macroF1=0.5642

MLP M0 per-class recall:
                  label   recall  support
       Uploading_Attack 0.010638      188
        Recon-PingSweep 0.008824 

In [ ]:
import importlib
from src import data, train, compression, models
for m in (data, train, compression): importlib.reload(m)
import pandas as pd, numpy as np, torch

# reload MLP anchor + rebuild its compression matrix
mlp, le, scaler, feat_cols = train.load_anchor("ciciot2023","mlp","M0",
                                               CFG["anchor_seed"], arch_kwargs={"hidden":(256,128)})
mat = compression.build_matrix(mlp, df, "ciciot2023", splits, seed=CFG["anchor_seed"],
                               arch="mlp", verbose=True)

# per-class recall for every cell
recs = {}
for cell, entry in mat.items():
    rec, macro, _, _, _ = compression.evaluate_cell(entry, df, splits, le, scaler, feat_cols)
    if rec is None:
        print(f"{cell}: SKIPPED ({entry['note']})"); continue
    recs[cell] = {le.classes_[c]: rec[c] for c in sorted(rec)}
    print(f"{cell:14} macroF1={macro:.4f}  sparsity={entry['size'].get('sparsity','')}")
R_mlp = pd.DataFrame(recs)
R_mlp.to_csv(PATHS.tables("compression","mlp_per_class_recall_matrix.csv"))

# Δrecall + collapse flags vs MLP's OWN bands... but MLP has no null band yet.
# For the cross-arch MECHANISM test we don't strictly need MLP's 5-seed band —
# we just need which classes collapsed, using a simple drop threshold for now.
delta_mlp = R_mlp.drop(columns=["M0"]).sub(R_mlp["M0"], axis=0)
# "collapsed" = lost >0.15 recall under prune80 (a clear, conservative drop)
prune80_collapsed = list(delta_mlp.index[delta_mlp["prune80"] < -0.15])
print("\nMLP prune80 macro-F1 drop and collapse:")
print(f"  classes losing >0.15 recall under prune80: {len(prune80_collapsed)}")
print(f"  {prune80_collapsed}")
delta_mlp.to_csv(PATHS.tables("compression","mlp_delta_recall.csv"))
print("\nsaved MLP matrix + delta -> results/tables/compression/")

[M0]
   uncompressed anchor | {'params': 48290, 'nonzero_params': 48290, 'sparsity': 0.0}
[prune50]
    prune50 ft epoch 0
    prune50 ft epoch 1
    prune50 ft epoch 2
    prune50 ft epoch 3
    prune50 ft epoch 4
    prune50 ft epoch 5
    prune50 ft epoch 6
    prune50 ft epoch 7
   L1 prune 50% + finetune | {'params': 48290, 'nonzero_params': 24738, 'sparsity': 0.4877}
[prune80]
    prune80 ft epoch 0
    prune80 ft epoch 1
    prune80 ft epoch 2
    prune80 ft epoch 3
    prune80 ft epoch 4
    prune80 ft epoch 5
    prune80 ft epoch 6
    prune80 ft epoch 7
   L1 prune 80% + finetune | {'params': 48290, 'nonzero_params': 10607, 'sparsity': 0.7803}
[distillation]


TypeError: MLP.__init__() got an unexpected keyword argument 'channels'

In [ ]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
src = open(f"{REPO}/src/compression.py").read()

# the distillation branch hardcodes channels=(24,48); make the student arch-aware
old = '''        elif cell == "distillation":
            sk = {"channels": (24, 48)}   # smaller student than the (64,128) anchor'''
new = '''        elif cell == "distillation":
            # student is a SMALLER version of the same architecture (arch-aware kwargs)
            sk = {"channels": (24, 48)} if arch in ("cnn1d", "gru") else \\
                 {"hidden": (96, 48)} if arch == "mlp" else \\
                 {"d_token": 16, "n_layers": 1}
'''
assert old in src, "anchor text not found — paste me the build_matrix distillation block"
src = src.replace(old, new)
open(f"{REPO}/src/compression.py","w").write(src)
print("patched build_matrix distillation to be arch-aware")
print("MLP student now uses hidden=(96,48)")

patched build_matrix distillation to be arch-aware
MLP student now uses hidden=(96,48)


In [ ]:
import importlib
from src import compression
importlib.reload(compression)
import pandas as pd, numpy as np

mat = compression.build_matrix(mlp, df, "ciciot2023", splits, seed=CFG["anchor_seed"],
                               arch="mlp", verbose=True)
recs = {}
for cell, entry in mat.items():
    rec, macro, _, _, _ = compression.evaluate_cell(entry, df, splits, le, scaler, feat_cols)
    if rec is None:
        print(f"{cell}: SKIPPED ({entry['note']})"); continue
    recs[cell] = {le.classes_[c]: rec[c] for c in sorted(rec)}
    print(f"{cell:14} macroF1={macro:.4f}  sparsity={entry['size'].get('sparsity','')}")
R_mlp = pd.DataFrame(recs)
R_mlp.to_csv(PATHS.tables("compression","mlp_per_class_recall_matrix.csv"))
delta_mlp = R_mlp.drop(columns=["M0"]).sub(R_mlp["M0"], axis=0)
prune80_collapsed = list(delta_mlp.index[delta_mlp["prune80"] < -0.15])
print(f"\nMLP prune80 collapsed (>0.15 drop): {len(prune80_collapsed)}")
print(f"  {prune80_collapsed}")
delta_mlp.to_csv(PATHS.tables("compression","mlp_delta_recall.csv"))
print("\nsaved MLP matrix + delta")

[M0]
   uncompressed anchor | {'params': 48290, 'nonzero_params': 48290, 'sparsity': 0.0}
[prune50]
    prune50 ft epoch 0
    prune50 ft epoch 1
    prune50 ft epoch 2
    prune50 ft epoch 3
    prune50 ft epoch 4
    prune50 ft epoch 5
    prune50 ft epoch 6
    prune50 ft epoch 7
   L1 prune 50% + finetune | {'params': 48290, 'nonzero_params': 24738, 'sparsity': 0.4877}
[prune80]
    prune80 ft epoch 0
    prune80 ft epoch 1
    prune80 ft epoch 2
    prune80 ft epoch 3
    prune80 ft epoch 4
    prune80 ft epoch 5
    prune80 ft epoch 6
    prune80 ft epoch 7
   L1 prune 80% + finetune | {'params': 48290, 'nonzero_params': 10607, 'sparsity': 0.7803}
[distillation]
    distill epoch 0
    distill epoch 1
    distill epoch 2
    distill epoch 3
    distill epoch 4
    distill epoch 5
    distill epoch 6
    distill epoch 7
    distill epoch 8
    distill epoch 9
    distill epoch 10
    distill epoch 11
   KD student (24,48) <- anchor | {'params': 10450, 'nonzero_params': 10450, 'spa

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "Path 3 (MLP): MAJOR architecture-contrast finding. MLP collapses ~2 classes under prune80 vs CNN's 10 (macroF1 holds 0.56) -> sink-consolidation is CNN-SPECIFIC, mechanism architecture-gated per prereg. int8 hurts MLP (full Linear quant, 0.36) but not CNN (partial); float16 preserves both. CNN conv-over-nonlocal-tabular-features = likely fragility cause"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   c5f746f..786ee06  main -> main



In [ ]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
ADD = r'''

def rank_collapse_analysis(model_M0, model_comp, df, splits, scaler, feat_cols, le,
                           is_half_comp=False, which="test", max_n=60000, seed=0):
    """Test whether compression collapses the REPRESENTATION rank (the entanglement
    story). Computes effective rank of the penultimate representation for M0 and the
    compressed model, GLOBALLY and per-class. If sink-consolidation = rank collapse,
    the compressed model's global + per-class effective rank drops sharply."""
    import numpy as np, pandas as pd
    rng = np.random.default_rng(seed)
    fM0, lM0, y = extract_features(model_M0, df, splits, scaler, feat_cols, le, which=which)
    fC, lC, _ = extract_features(model_comp, df, splits, scaler, feat_cols, le,
                                 which=which, is_half=is_half_comp)
    if len(y) > max_n:
        idx = rng.choice(len(y), max_n, replace=False)
        fM0s, fCs, ys = fM0[idx], fC[idx], y[idx]
    else:
        fM0s, fCs, ys = fM0, fC, y
    glob = {
        "M0_effrank": geom.effective_rank(fM0s),
        "comp_effrank": geom.effective_rank(fCs),
        "M0_dim": fM0s.shape[1],
    }
    glob["rank_retained_frac"] = round(glob["comp_effrank"] / glob["M0_effrank"], 3)
    rows = []
    for c in np.unique(ys):
        m = ys == c
        if m.sum() < 20:
            continue
        rows.append({
            "label": le.classes_[int(c)],
            "support": int(m.sum()),
            "M0_class_effrank": round(geom.effective_rank(fM0s[m]), 3),
            "comp_class_effrank": round(geom.effective_rank(fCs[m]), 3),
        })
    pc = pd.DataFrame(rows)
    pc["rank_change"] = (pc["comp_class_effrank"] - pc["M0_class_effrank"]).round(3)
    return glob, pc.sort_values("support")
'''
with open(f"{REPO}/src/explain.py","a") as f:
    f.write(ADD)
print("appended rank_collapse_analysis")

appended rank_collapse_analysis


In [ ]:
# reload explain (file-path bypass) to get the new function
import importlib.util, sys
spec = importlib.util.spec_from_file_location("src.explain",
        "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/src/explain.py")
explain = importlib.util.module_from_spec(spec); sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)

import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd, numpy as np

# --- CNN: anchor + prune80 ---
cnn, cle, cscaler, cfeat = train.load_anchor("ciciot2023","cnn1d","M0",CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
cnn_p80, _, _ = compression.prune_and_finetune(cnn, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
print("CNN M0+prune80 ready")

# --- MLP: anchor + prune80 ---
mlp, mle, mscaler, mfeat = train.load_anchor("ciciot2023","mlp","M0",CFG["anchor_seed"], arch_kwargs={"hidden":(256,128)})
mlp_p80, _, _ = compression.prune_and_finetune(mlp, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="mlp", verbose=False)
print("MLP M0+prune80 ready")

# --- rank-collapse analysis for each ---
print("\n" + "="*70)
print("CNN: effective rank, M0 vs prune80")
gC, pcC = explain.rank_collapse_analysis(cnn, cnn_p80, df, splits, cscaler, cfeat, cle)
print(f"  GLOBAL: M0 effrank={gC['M0_effrank']:.2f} -> prune80={gC['comp_effrank']:.2f}  (retained {gC['rank_retained_frac']}, dim={gC['M0_dim']})")

print("\nMLP: effective rank, M0 vs prune80")
gM, pcM = explain.rank_collapse_analysis(mlp, mlp_p80, df, splits, mscaler, mfeat, mle)
print(f"  GLOBAL: M0 effrank={gM['M0_effrank']:.2f} -> prune80={gM['comp_effrank']:.2f}  (retained {gM['rank_retained_frac']}, dim={gM['M0_dim']})")

print("\n" + "="*70)
print("THE TEST: does the CNN's global rank collapse while the MLP's holds?")
print(f"  CNN rank retained: {gC['rank_retained_frac']}   MLP rank retained: {gM['rank_retained_frac']}")

# per-class rank change, worst-collapsing CNN classes
print("\nCNN per-class effective rank (M0 -> prune80), lowest support first:")
print(pcC.to_string(index=False))

pd.DataFrame([{"arch":"cnn", **gC},{"arch":"mlp", **gM}]).to_csv(PATHS.tables("explain","rank_collapse_global.csv"), index=False)
pcC.to_csv(PATHS.tables("explain","cnn_per_class_rank.csv"), index=False)
pcM.to_csv(PATHS.tables("explain","mlp_per_class_rank.csv"), index=False)
print("\nsaved rank analysis -> results/tables/explain/")

CNN M0+prune80 ready
MLP M0+prune80 ready

CNN: effective rank, M0 vs prune80
  GLOBAL: M0 effrank=72.61 -> prune80=55.24  (retained 0.761, dim=128)

MLP: effective rank, M0 vs prune80
  GLOBAL: M0 effrank=84.13 -> prune80=85.25  (retained 1.013, dim=128)

THE TEST: does the CNN's global rank collapse while the MLP's holds?
  CNN rank retained: 0.761   MLP rank retained: 1.013

CNN per-class effective rank (M0 -> prune80), lowest support first:
                  label  support  M0_class_effrank  comp_class_effrank  rank_change
       Uploading_Attack       23            17.192              15.838       -1.354
        Recon-PingSweep       41            29.595              26.361       -3.234
                    XSS       67            41.214              32.847       -8.367
       Backdoor_Malware       70            43.126              33.691       -9.435
           SqlInjection       93            51.305              41.626       -9.679
       CommandInjection       95            32.

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "EXPLAIN Test 1 (rank collapse): CONFIRMED. CNN retains 0.76 effective rank under prune80 (72.6->55.2) while MLP retains 1.01 (stable). Per-class rank loss tracks sink-consolidation victims (DictBruteForce -32, PortScan/OSScan/HostDiscovery ~-24, all sink-absorbed). Mechanism: conv-over-nonlocal-features -> rank collapse -> class subspaces crushed -> sink consolidation; MLP preserves rank -> robust"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   786ee06..0da888c  main -> main



In [ ]:
# Computes trust metrics — refuse to run until the prereg is frozen.
require_frozen()

In [ ]:
# ── FULL bootstrap ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch, pandas as pd, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4


In [ ]:
# data reload
from src import data
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("rows:", len(df))

rows: 3661696


In [ ]:
# reload explain (file-path bypass) + modules
import importlib.util, sys
spec = importlib.util.spec_from_file_location("src.explain", f"{REPO}/src/explain.py")
explain = importlib.util.module_from_spec(spec); sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd, numpy as np

# MLP anchor + prune80
mlp, mle, mscaler, mfeat = train.load_anchor("ciciot2023","mlp","M0",CFG["anchor_seed"], arch_kwargs={"hidden":(256,128)})
mlp_p80, _, _ = compression.prune_and_finetune(mlp, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="mlp", verbose=False)
print("MLP M0+prune80 ready")

# which MLP classes collapsed under prune80 (from saved delta, >0.15 drop)
delta_mlp = pd.read_csv(PATHS.tables("compression","mlp_delta_recall.csv"), index_col=0)
mlp_collapsed = list(delta_mlp.index[delta_mlp["prune80"] < -0.15])
print("MLP prune80 collapsed:", mlp_collapsed)

# confusability: where do the MLP's collapsed classes' samples go?
conf_mlp = explain.confusability_reallocation(mlp, mlp_p80, df, splits, mscaler, mfeat, mle, mlp_collapsed)
print("\n=== MLP CONFUSABILITY: where did collapsed classes' samples go? ===")
print(conf_mlp.to_string(index=False))
conf_mlp.to_csv(PATHS.tables("explain","mlp_prune80_confusability.csv"), index=False)
print("\nsaved -> results/tables/explain/mlp_prune80_confusability.csv")

MLP M0+prune80 ready
MLP prune80 collapsed: ['DDoS-PSHACK_Flood', 'DDoS-TCP_Flood']

=== MLP CONFUSABILITY: where did collapsed classes' samples go? ===
  collapsed_class  M0_recall  comp_recall   top_absorber  absorber_took_frac  absorber_recall_M0  absorber_recall_comp  absorber_recall_gain
   DDoS-TCP_Flood      0.664        0.398  DoS-TCP_Flood                0.60               0.556                 0.803                 0.248
DDoS-PSHACK_Flood      0.998        0.689 DDoS-SlowLoris                0.31               0.962                 0.971                 0.009

saved -> results/tables/explain/mlp_prune80_confusability.csv


In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "MLP confusability: NO sink (2 collapses -> 2 different absorbers, isolated confusable pairs) vs CNN's single VulnScan sink absorbing 10. Architecture contrast complete: rank preserved -> no subspace crush -> no consolidation. Mechanism coherent across both architectures"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   0da888c..dd89e5b  main -> main



In [ ]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/shuffle_experiment.py — Test 3: is the CNN's pruning fragility FEATURE-ORDER
dependent (=> convolution-over-non-local-tabular-features is the root cause),
while the MLP is order-invariant (control)?
"""
from __future__ import annotations
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import f1_score

from .config import CFG, PATHS
from . import train as _train
from . import compression as _comp
from . import explain as _explain
from . import geometry as _geom


def permute_features(df, feat_cols, seed):
    """Return a copy of df with feature columns reordered by a fixed permutation.
    Renames to feat_0..feat_N so column NAMES are identical, DATA order differs."""
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(feat_cols))
    new_order = [feat_cols[i] for i in perm]
    meta = [c for c in df.columns if c not in feat_cols]
    out = df[new_order + meta].copy()
    rename = {new_order[i]: f"feat_{i}" for i in range(len(new_order))}
    out = out.rename(columns=rename)
    return out, [f"feat_{i}" for i in range(len(new_order))], perm.tolist()


def run_one_ordering(arch, df, splits, seed, order_seed, arch_kwargs, *,
                     collapse_thresh=0.15, ft_epochs=8, epochs=40, patience=6):
    """Train arch on ONE feature ordering, prune80, return collapse summary."""
    feat_cols = _train.feature_columns(df)
    dfp, pcols, perm = permute_features(df, feat_cols, order_seed)

    model, info = _train.train_model(arch, dfp, "ciciot2023_shuf", splits, seed,
                                     epochs=epochs, patience=patience,
                                     arch_kwargs=arch_kwargs, save=False, verbose=False)
    le, scaler = info["label_encoder"], info["scaler"]

    yt, yp, _ = _train.predict(model, dfp, splits, le, scaler, pcols)
    r0 = _train.per_class_recall_table(yt, yp, le).set_index("label")["recall"]

    p80, _le, _sc = _comp.prune_and_finetune(model, dfp, "ciciot2023_shuf", splits, seed,
                                             0.80, ft_epochs=ft_epochs, arch=arch, verbose=False)
    ytp, ypp, _ = _train.predict(p80, dfp, splits, le, scaler, pcols)
    r8 = _train.per_class_recall_table(ytp, ypp, le).set_index("label")["recall"]
    macro8 = f1_score(ytp, ypp, average="macro")
    macro0 = info["macroF1_test"]

    delta = (r8 - r0)
    collapsed = set(delta.index[delta < -collapse_thresh])

    gC, _ = _explain.rank_collapse_analysis(model, p80, dfp, splits, scaler, pcols, le, max_n=40000)

    return {
        "order_seed": order_seed,
        "macroF1_M0": round(macro0, 4),
        "macroF1_prune80": round(macro8, 4),
        "n_collapsed": len(collapsed),
        "collapsed_set": collapsed,
        "rank_retained": gC["rank_retained_frac"],
        "delta_recall": delta,
    }


def feature_order_sensitivity(arch, df, splits, seed, order_seeds, arch_kwargs, **kw):
    """Run multiple orderings; quantify how much the collapse pattern VARIES.
    Returns (summary_df, jaccard_stability, collapsed_union, runs)."""
    runs = [run_one_ordering(arch, df, splits, seed, os_, arch_kwargs, **kw) for os_ in order_seeds]
    rows = [{k: v for k, v in r.items() if k not in ("collapsed_set", "delta_recall")} for r in runs]
    summ = pd.DataFrame(rows)
    sets = [r["collapsed_set"] for r in runs]
    jac = []
    for i in range(len(sets)):
        for j in range(i + 1, len(sets)):
            u = sets[i] | sets[j]; inter = sets[i] & sets[j]
            jac.append(len(inter) / len(u) if u else 1.0)
    jaccard_stability = float(np.mean(jac)) if jac else 1.0
    union = set().union(*sets) if sets else set()
    return summ, jaccard_stability, union, runs
'''
open(f"{REPO}/src/shuffle_experiment.py","w").write(SRC)
print("wrote shuffle_experiment.py:", len(SRC), "chars")
assert "feature_order_sensitivity" in SRC and "permute_features" in SRC
print("OK — shuffle_experiment.py written")

wrote shuffle_experiment.py: 3693 chars
OK — shuffle_experiment.py written


In [ ]:
import importlib.util, sys
# load shuffle_experiment + fresh explain via file-path
for name in ["src.explain","src.shuffle_experiment"]:
    fn = name.split(".")[1]
    spec = importlib.util.spec_from_file_location(name, f"{REPO}/src/{fn}.py")
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod; spec.loader.exec_module(mod)
import src.shuffle_experiment as shuf
import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd

ORDER_SEEDS = [101, 202, 303]   # 3 distinct feature-column permutations

print("="*70); print("CNN: feature-order sensitivity (3 orderings)")
cnn_summ, cnn_jac, cnn_union, cnn_runs = shuf.feature_order_sensitivity(
    "cnn1d", df, splits, CFG["anchor_seed"], ORDER_SEEDS, {"channels":(64,128)})
print(cnn_summ.to_string(index=False))
print(f"CNN collapsed-set Jaccard stability: {cnn_jac:.3f}  (1.0=same victims every ordering, low=order-sensitive)")
print(f"CNN rank_retained across orderings: {[r['rank_retained'] for r in cnn_runs]}")
print(f"CNN collapsed union: {sorted(cnn_union)}")

print("\n"+"="*70); print("MLP: feature-order sensitivity (3 orderings) — CONTROL")
mlp_summ, mlp_jac, mlp_union, mlp_runs = shuf.feature_order_sensitivity(
    "mlp", df, splits, CFG["anchor_seed"], ORDER_SEEDS, {"hidden":(256,128)})
print(mlp_summ.to_string(index=False))
print(f"MLP collapsed-set Jaccard stability: {mlp_jac:.3f}")
print(f"MLP rank_retained across orderings: {[r['rank_retained'] for r in mlp_runs]}")

print("\n"+"="*70); print("VERDICT")
print(f"  CNN macroF1 across orderings: {cnn_summ['macroF1_M0'].tolist()} (M0), {cnn_summ['macroF1_prune80'].tolist()} (prune80)")
print(f"  MLP macroF1 across orderings: {mlp_summ['macroF1_M0'].tolist()} (M0), {mlp_summ['macroF1_prune80'].tolist()} (prune80)")
print(f"  CNN n_collapsed varies: {cnn_summ['n_collapsed'].tolist()}  | MLP: {mlp_summ['n_collapsed'].tolist()}")

cnn_summ.to_csv(PATHS.tables("explain","cnn_feature_order_sensitivity.csv"), index=False)
mlp_summ.to_csv(PATHS.tables("explain","mlp_feature_order_sensitivity.csv"), index=False)
print("\nsaved -> results/tables/explain/")

CNN: feature-order sensitivity (3 orderings)
 order_seed  macroF1_M0  macroF1_prune80  n_collapsed  rank_retained
        101      0.5353           0.1048           21          0.738
        202      0.5249           0.1774           20          0.674
        303      0.5294           0.2351           18          0.713
CNN collapsed-set Jaccard stability: 0.582  (1.0=same victims every ordering, low=order-sensitive)
CNN rank_retained across orderings: [0.738, 0.674, 0.713]
CNN collapsed union: ['BenignTraffic', 'DDoS-ACK_Fragmentation', 'DDoS-HTTP_Flood', 'DDoS-ICMP_Flood', 'DDoS-ICMP_Fragmentation', 'DDoS-PSHACK_Flood', 'DDoS-RSTFINFlood', 'DDoS-SYN_Flood', 'DDoS-SlowLoris', 'DDoS-SynonymousIP_Flood', 'DDoS-TCP_Flood', 'DDoS-UDP_Flood', 'DDoS-UDP_Fragmentation', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-TCP_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Mirai-greeth_flood', 'Mirai-greip_flood', 'Mirai-udpplain', 'Recon-HostDiscovery', 'Recon-OSScan

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "EXPLAIN Test 3 (feature-order sensitivity): ROOT CAUSE established. CNN prune80 macroF1 swings 2.2x across feature orderings (0.105-0.235), rank collapse varies (0.67-0.74), collapsed-set Jaccard 0.58; MLP INVARIANT (macroF1 flat ~0.55, rank ~1.0, 0-2 collapses) -> permutation-invariance control. Convolution over non-local tabular features is causal root of CNN pruning fragility. (MLP Jaccard 0.0 is small-set artifact, report via magnitudes)"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   dd89e5b..252a54a  main -> main



In [ ]:
print("ready:", all(x in dir() for x in ['df','splits','CFG','PATHS','train']),
      "| rows:", len(df) if 'df' in dir() else "—")

ready: False | rows: 3661696


In [ ]:
from src import data, train
print("now ready:", all(x in dir() for x in ['df','splits','CFG','PATHS','train']))

now ready: False


In [ ]:
from src.config import CFG, PATHS, set_all_seeds
print("ready check:", all(x in dir() for x in ['df','splits','CFG','PATHS','train']))

ready check: False


In [ ]:
for name in ['df','splits','CFG','PATHS','train']:
    print(f"  {name}: {'present' if name in dir() else 'MISSING'}")

  df: present
  splits: present
  CFG: present
  PATHS: present
  train: present


In [ ]:
# ── FULL bootstrap ──
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch, pandas as pd, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: Tesla T4


In [ ]:
from src import data, train
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("rows:", len(df))

rows: 3661696


In [ ]:
import importlib
from src import data, train
importlib.reload(train)

ftt, ftt_info = train.train_model(
    "ft_transformer", df, "ciciot2023", splits, seed=CFG["anchor_seed"],
    epochs=40, patience=6, batch_size=4096, arch_kwargs={"d_token": 64, "n_heads": 4, "n_layers": 2})
print(f"\nFT-Transformer params={ftt_info['params']}  val_macroF1={ftt_info['macroF1_val']:.4f}  test_macroF1={ftt_info['macroF1_test']:.4f}")

yt, yp, probs = train.predict(ftt, df, splits, ftt_info["label_encoder"], ftt_info["scaler"], ftt_info["feat_cols"])
tab = train.per_class_recall_table(yt, yp, ftt_info["label_encoder"])
print("\nFT-Transformer M0 per-class recall:")
print(tab.to_string(index=False))
tab.to_csv(PATHS.tables("baseline","ftt_M0_per_class_recall.csv"), index=False)

  epoch 00  val_macroF1=0.4228
  epoch 01  val_macroF1=0.4563
  epoch 02  val_macroF1=0.4601
  epoch 03  val_macroF1=0.4945
  epoch 04  val_macroF1=0.4997
  epoch 05  val_macroF1=0.4926
  epoch 06  val_macroF1=0.4897
  epoch 07  val_macroF1=0.4890
  epoch 08  val_macroF1=0.4901
  epoch 09  val_macroF1=0.5153
  epoch 10  val_macroF1=0.4963
  epoch 11  val_macroF1=0.4957
  epoch 12  val_macroF1=0.5153
  epoch 13  val_macroF1=0.5070
  epoch 14  val_macroF1=0.5085
  epoch 15  val_macroF1=0.5151
  early stop @ epoch 15
  saved -> /content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression/models/ciciot2023/ciciot2023__ft_transformer__M0__seed0.pt

FT-Transformer params=69346  val_macroF1=0.5153  test_macroF1=0.4661

FT-Transformer M0 per-class recall:
                  label   recall  support
       Uploading_Attack 0.000000      188
        Recon-PingSweep 0.000000      340
       Backdoor_Malware 0.000000      483
                    XSS 0.000000      577
           SqlInjection 0.000

In [ ]:
import importlib
from src import train
importlib.reload(train)

# transformers train slower + bumpier: more epochs, more patience, bigger model, warmup-friendly LR
ftt, ftt_info = train.train_model(
    "ft_transformer", df, "ciciot2023", splits, seed=CFG["anchor_seed"],
    epochs=80, patience=15, batch_size=4096, lr=5e-4,
    arch_kwargs={"d_token": 96, "n_heads": 8, "n_layers": 3})
print(f"\nFT-Transformer params={ftt_info['params']}  val_macroF1={ftt_info['macroF1_val']:.4f}  test_macroF1={ftt_info['macroF1_test']:.4f}")

yt, yp, probs = train.predict(ftt, df, splits, ftt_info["label_encoder"], ftt_info["scaler"], ftt_info["feat_cols"])
tab = train.per_class_recall_table(yt, yp, ftt_info["label_encoder"])
print("\nFT-Transformer M0 per-class recall:")
print(tab.to_string(index=False))
tab.to_csv(PATHS.tables("baseline","ftt_M0_per_class_recall.csv"), index=False)

  epoch 00  val_macroF1=0.4276
  epoch 01  val_macroF1=0.4534
  epoch 02  val_macroF1=0.4738
  epoch 03  val_macroF1=0.4802
  epoch 04  val_macroF1=0.4892
  epoch 05  val_macroF1=0.4981
  epoch 06  val_macroF1=0.4942
  epoch 07  val_macroF1=0.5030
  epoch 08  val_macroF1=0.4938
  epoch 09  val_macroF1=0.5083
  epoch 10  val_macroF1=0.5018
  epoch 11  val_macroF1=0.5064
  epoch 12  val_macroF1=0.5069
  epoch 13  val_macroF1=0.5097
  epoch 14  val_macroF1=0.5223
  epoch 15  val_macroF1=0.5193
  epoch 16  val_macroF1=0.5145
  epoch 17  val_macroF1=0.5078
  epoch 18  val_macroF1=0.5148
  epoch 19  val_macroF1=0.5219
  epoch 20  val_macroF1=0.5065
  epoch 21  val_macroF1=0.5254
  epoch 22  val_macroF1=0.5224
  epoch 23  val_macroF1=0.5178
  epoch 24  val_macroF1=0.5241
  epoch 25  val_macroF1=0.5331
  epoch 26  val_macroF1=0.5241
  epoch 27  val_macroF1=0.5289
  epoch 28  val_macroF1=0.5324
  epoch 29  val_macroF1=0.5336
  epoch 30  val_macroF1=0.5264
  epoch 31  val_macroF1=0.5261
  epoch 

In [ ]:
# bootstrap, then verify the transformer checkpoint loads
from google.colab import drive; drive.mount('/content/drive')
import os, sys
REPO='/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0,REPO)
from src.config import CFG, PATHS
from src import train
import torch
ftt, le, scaler, feat = train.load_anchor("ciciot2023","ft_transformer","M0",CFG["anchor_seed"],
                                           arch_kwargs={"d_token":96,"n_heads":8,"n_layers":3})
print("transformer loaded:", ftt.num_params(), "params")

Mounted at /content/drive
transformer loaded: 227938 params


In [ ]:
import json, os
ftt_record = {
    "arch": "ft_transformer_lite",
    "params": 227938,
    "test_macroF1": 0.5081,
    "val_macroF1": 0.5445,
    "cnn_test_macroF1": 0.545,
    "mlp_test_macroF1": 0.564,
    "training_time_hours": 4.5,
    "epochs": 80,
    "config": {"d_token": 96, "n_heads": 8, "n_layers": 3},
    "exclusion_reason": "Failed matched-baseline (0.51 vs 0.55/0.56) despite 4-7x params and 4.5h training; "
                        "excluded from primary architecture comparison to avoid baseline-mismatch confound; "
                        "violates the edge-scale envelope (227k params, 4.5h) the study targets; "
                        "consistent with known transformer underperformance on tabular data at this scale.",
    "decision": "Report CNN + MLP as primary 2-architecture contrast; FT-Transformer noted as supplementary exclusion."
}
with open(f"{REPO}/results/tables/explain/ftt_exclusion_record.json", "w") as f:
    json.dump(ftt_record, f, indent=2)
print("recorded FT-Transformer exclusion rationale")
print(json.dumps(ftt_record, indent=2))

recorded FT-Transformer exclusion rationale
{
  "arch": "ft_transformer_lite",
  "params": 227938,
  "test_macroF1": 0.5081,
  "val_macroF1": 0.5445,
  "cnn_test_macroF1": 0.545,
  "mlp_test_macroF1": 0.564,
  "training_time_hours": 4.5,
  "epochs": 80,
  "config": {
    "d_token": 96,
    "n_heads": 8,
    "n_layers": 3
  },
  "exclusion_reason": "Failed matched-baseline (0.51 vs 0.55/0.56) despite 4-7x params and 4.5h training; excluded from primary architecture comparison to avoid baseline-mismatch confound; violates the edge-scale envelope (227k params, 4.5h) the study targets; consistent with known transformer underperformance on tabular data at this scale.",
  "decision": "Report CNN + MLP as primary 2-architecture contrast; FT-Transformer noted as supplementary exclusion."
}


In [ ]:
import subprocess
os.chdir(REPO)
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "Stage 2 CLOSED: FT-Transformer excluded from primary comparison (0.51 test vs 0.55/0.56 CNN/MLP, 227k params, 4.5h train -> fails matched-baseline + violates edge-scale envelope). Documented reasoned prereg deviation. EXPLAIN = clean 2-arch contrast (CNN fragile via rank-collapse/sink, MLP robust) + Test1 rank evidence + Test3 root-cause."
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   252a54a..63d51db  main -> main



In [ ]:
import os
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
for sub in ["results/tables/baseline","results/tables/compression","results/tables/explain"]:
    print(f"\n{sub}:")
    p = f"{REPO}/{sub}"
    if os.path.isdir(p):
        for f in sorted(os.listdir(p)): print("  ", f)


results/tables/baseline:
   cnn1d_M0_null_band.csv
   cnn1d_M0_null_band_5seed.csv
   cnn1d_M0_per_class_recall.csv
   cnn1d_M0_tiers.csv
   ftt_M0_per_class_recall.csv
   mlp_M0_per_class_recall.csv

results/tables/compression:
   cnn1d_collapse_flags.csv
   cnn1d_delta_recall.csv
   cnn1d_per_class_recall_matrix.csv
   mlp_delta_recall.csv
   mlp_per_class_recall_matrix.csv

results/tables/explain:
   cnn1d_M0_geometry.csv
   cnn1d_prune80_geometry.csv
   cnn_feature_order_sensitivity.csv
   cnn_per_class_rank.csv
   ftt_exclusion_record.json
   geometry_vs_collapse.csv
   mlp_feature_order_sensitivity.csv
   mlp_per_class_rank.csv
   mlp_prune80_confusability.csv
   prune80_confusability.csv
   rank_collapse_global.csv


In [ ]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
ADD = r'''

def crux_probe(model_M0, model_comp, df, splits, scaler, feat_cols, le,
               is_half_comp=False, which="test", max_n=40000, seed=0, C=1.0):
    """THE CRUX (plan D1): per-class one-vs-rest linear probe on the FROZEN penultimate
    representation of M0 vs compressed; compare AUC.
    Survives -> info still decodable, collapse is decision-layer (re-thresholding fixes).
    Collapses -> info genuinely lost, true capacity loss (must touch compression)."""
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split
    rng = np.random.default_rng(seed)
    fM0, lM0, y = extract_features(model_M0, df, splits, scaler, feat_cols, le, which=which)
    fC, lC, _ = extract_features(model_comp, df, splits, scaler, feat_cols, le,
                                 which=which, is_half=is_half_comp)
    if len(y) > max_n:
        idx = rng.choice(len(y), max_n, replace=False)
        fM0, fC, y = fM0[idx], fC[idx], y[idx]
    tr, te = train_test_split(np.arange(len(y)), test_size=0.4, random_state=seed, stratify=y)
    def probe_auc(feats, c):
        yb = (y == c).astype(int)
        if yb[tr].sum() < 5 or yb[te].sum() < 2:
            return np.nan
        clf = LogisticRegression(max_iter=500, C=C, class_weight="balanced")
        clf.fit(feats[tr], yb[tr])
        return roc_auc_score(yb[te], clf.predict_proba(feats[te])[:, 1])
    rows = []
    for c in np.unique(y):
        a0 = probe_auc(fM0, c); ac = probe_auc(fC, c)
        rows.append({
            "label": le.classes_[int(c)],
            "support_sub": int((y == c).sum()),
            "M0_probe_auc": round(a0, 4) if a0 == a0 else np.nan,
            "comp_probe_auc": round(ac, 4) if ac == ac else np.nan,
            "probe_auc_drop": round(a0 - ac, 4) if (a0 == a0 and ac == ac) else np.nan,
        })
    return pd.DataFrame(rows).set_index("label")
'''
with open(f"{REPO}/src/explain.py","a") as f:
    f.write(ADD)
print("appended crux_probe")

appended crux_probe


In [ ]:
# bootstrap if needed
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
if 'CFG' not in dir():
    from google.colab import drive; drive.mount('/content/drive')
    os.chdir(REPO); sys.path.insert(0, REPO)
    from src.config import CFG, PATHS, set_all_seeds
    import torch, pandas as pd, numpy as np
if 'df' not in dir():
    from src import data
    df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
    splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
    print("data reloaded, rows:", len(df))

# load explain (file-path) + rebuild CNN M0 + prune80
import importlib.util
spec = importlib.util.spec_from_file_location("src.explain", f"{REPO}/src/explain.py")
explain = importlib.util.module_from_spec(spec); sys.modules["src.explain"] = explain
spec.loader.exec_module(explain)
import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd

cnn, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
cnn_p80, _, _ = compression.prune_and_finetune(cnn, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
print("CNN M0+prune80 ready\n")

# THE CRUX
crux = explain.crux_probe(cnn, cnn_p80, df, splits, scaler, feat_cols, le)

# attach recall change + tier for interpretation
delta = pd.read_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"), index_col=0)["prune80"]
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]
crux["prune80_d_recall"] = delta
crux["tier"] = tiers
crux_meas = crux[crux["tier"]=="measurable"].sort_values("prune80_d_recall")

print("="*90)
print("CRUX: probe AUC (M0 vs prune80) for the COLLAPSED measurable classes")
print("  AUC survives + recall drops = DECISION-LAYER collapse (info survived, re-threshold fixes)")
print("  AUC drops + recall drops    = INFORMATION LOSS (true capacity loss)")
print("="*90)
print(crux_meas.to_string())

# the verdict: among classes that LOST recall, did the probe survive?
collapsed = crux_meas[crux_meas["prune80_d_recall"] < -0.10]
print(f"\nAmong {len(collapsed)} classes that lost >0.10 recall under prune80:")
print(f"  mean probe AUC drop: {collapsed['probe_auc_drop'].mean():.4f}")
print(f"  mean M0 probe AUC:   {collapsed['M0_probe_auc'].mean():.4f}")
print(f"  mean prune80 AUC:    {collapsed['comp_probe_auc'].mean():.4f}")
print(f"  -> classes keeping AUC>0.85 despite recall loss: {(collapsed['comp_probe_auc']>0.85).sum()}/{len(collapsed)}")

crux.to_csv(PATHS.tables("explain","cnn_crux_probe_prune80.csv"))
print("\nsaved -> results/tables/explain/cnn_crux_probe_prune80.csv")

data reloaded, rows: 3661696
CNN M0+prune80 ready

CRUX: probe AUC (M0 vs prune80) for the COLLAPSED measurable classes
  AUC survives + recall drops = DECISION-LAYER collapse (info survived, re-threshold fixes)
  AUC drops + recall drops    = INFORMATION LOSS (true capacity loss)
                      support_sub  M0_probe_auc  comp_probe_auc  probe_auc_drop  prune80_d_recall        tier
label                                                                                                        
DoS-UDP_Flood                1647        0.9927          0.9928         -0.0001         -0.682311  measurable
DoS-HTTP_Flood                796        0.9975          0.9976         -0.0002         -0.654015  measurable
Recon-HostDiscovery          1410        0.9806          0.9712          0.0094         -0.628807  measurable
BenignTraffic                2245        0.9701          0.9640          0.0061         -0.500584  measurable
Recon-PortScan                816        0.9314          0

In [ ]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "CRUX experiment (plan D1, the spine): DECISION-LAYER verdict. Collapsed classes retain near-baseline linear-probe AUC (mean drop 0.009, 10/10 keep AUC>0.85) despite recall->0 (DoS-UDP 0.68->0 recall, probe AUC unchanged 0.993). Information PRESERVED; collapse is decision-boundary reallocation to confusable neighbors, not capacity loss. => MITIGATE can be near-free re-thresholding; PREDICT should weight margin/confusability features. Mechanism fully resolved: rank-collapse + info-preserved + decision-layer + confusability-sink"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   63d51db..d32669f  main -> main



In [ ]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/predict.py — Stage 3: PREDICT (the headline).
Forecast which classes lose recall under compression, from the M0 model + data ALONE.
Option 3: assemble neural-collapse + confusability + frequency/capacity features, let
the data adjudicate. Tiny-n discipline: low-capacity rules, LOCO-CV, bootstrap CIs.
Must beat frequency-only AND margin-only (Tran & Fioretto) baselines.
"""
from __future__ import annotations
import numpy as np
import pandas as pd
import torch
from scipy.stats import spearmanr, kendalltau

from .config import CFG, PATHS
from . import explain as _explain


def assemble_features(model_M0, df, splits, scaler, feat_cols, le, which="test",
                      max_n=60000, seed=0):
    """All candidate per-class predictor features from the M0 model only."""
    feats, logits, y = _explain.extract_features(model_M0, df, splits, scaler, feat_cols, le, which=which)
    rng = np.random.default_rng(seed)
    if len(y) > max_n:
        idx = rng.choice(len(y), max_n, replace=False)
        feats, logits, y = feats[idx], logits[idx], y[idx]
    classes = np.unique(y)
    names = [le.classes_[int(c)] for c in classes]
    pred = logits.argmax(1)
    probs = np.exp(logits - logits.max(1, keepdims=True)); probs /= probs.sum(1, keepdims=True)
    means = {int(c): feats[y == c].mean(0) for c in classes}
    support = {int(c): int((y == c).sum()) for c in classes}
    K = len(le.classes_)
    cm = np.zeros((K, K))
    for t, p in zip(y, pred):
        cm[t, p] += 1
    cm_norm = cm / cm.sum(1, keepdims=True).clip(min=1)
    etf = _explain.geom.etf_deviation(feats, y)
    marg = _explain.geom.per_class_margin(logits, y)
    rows = {}
    for c in classes:
        ci = int(c); m_c = means[ci]
        higher = [int(o) for o in classes if support[int(o)] > support[ci]]
        if higher:
            sims = [np.dot(m_c, means[o]) / (np.linalg.norm(m_c)*np.linalg.norm(means[o]) + 1e-9)
                    for o in higher]
            sim_nearest_higher = float(np.max(sims))
        else:
            sim_nearest_higher = 0.0   # most-frequent class: upward-confusability is zero, not missing
        offdiag = float(cm_norm[ci].sum() - cm_norm[ci, ci])
        pc = probs[y == c].mean(0); pc = pc[pc > 0]
        pred_entropy = float(-(pc * np.log(pc)).sum())
        rows[names[list(classes).index(c)]] = {
            "support": support[ci],
            "baseline_recall": float((pred[y == c] == c).mean()),
            "etf_cos_to_others": etf[c]["mean_cos_to_others"],
            "margin": marg[c],
            "sim_nearest_higher_freq": sim_nearest_higher,
            "confusion_offdiag": offdiag,
            "pred_entropy": pred_entropy,
        }
    return pd.DataFrame(rows).T


def build_prediction_table(feature_df, delta_recall_df, cells=("prune50","prune80","distillation"),
                           tiers=None, measurable_only=False):
    idx = feature_df.index
    if measurable_only and tiers is not None:
        idx = [c for c in idx if tiers.get(c) == "measurable"]
    recs = []
    for cls in idx:
        if cls not in delta_recall_df.index:
            continue
        for cell in cells:
            if cell not in delta_recall_df.columns:
                continue
            row = dict(feature_df.loc[cls])
            row["class"] = cls; row["cell"] = cell
            row["delta_recall"] = float(delta_recall_df.loc[cls, cell])
            if tiers is not None:
                row["tier"] = tiers.get(cls, "NA")
            recs.append(row)
    return pd.DataFrame(recs)


def screen_features(table, feature_cols, target="delta_recall"):
    rows = []
    d = table[target].astype(float)
    for f in feature_cols:
        x = table[f].astype(float)
        ok = x.notna() & d.notna()
        if ok.sum() < 5:
            rows.append({"feature": f, "spearman": np.nan, "n": int(ok.sum())}); continue
        sr, sp = spearmanr(x[ok], d[ok]); kt, kp = kendalltau(x[ok], d[ok])
        rows.append({"feature": f, "spearman": round(sr,4), "spearman_p": round(sp,4),
                     "kendall": round(kt,4), "n": int(ok.sum())})
    return pd.DataFrame(rows).sort_values("spearman", key=lambda s: s.abs(), ascending=False)


def _loco_predict(table, feature_cols, target="delta_recall"):
    from sklearn.linear_model import LinearRegression
    classes = table["class"].unique()
    X = table[feature_cols].astype(float)
    valid = X.notna().all(1)
    t = table[valid].reset_index(drop=True)
    X = t[feature_cols].astype(float).values; y = t[target].astype(float).values
    cls_arr = t["class"].values
    preds, actuals, keys = [], [], []
    for held in classes:
        tr = cls_arr != held; te = cls_arr == held
        if tr.sum() < 5 or te.sum() == 0:
            continue
        m = LinearRegression().fit(X[tr], y[tr]); p = m.predict(X[te])
        preds.extend(p); actuals.extend(y[te]); keys.extend([held]*te.sum())
    return np.array(preds), np.array(actuals), keys


def evaluate_predictor(table, feature_sets, target="delta_recall", n_boot=1000, seed=0):
    from sklearn.metrics import r2_score
    rng = np.random.default_rng(seed); results = {}
    for name, cols in feature_sets.items():
        preds, actuals, keys = _loco_predict(table, cols, target)
        if len(preds) < 5:
            results[name] = {"error": "insufficient data"}; continue
        sr, sp = spearmanr(preds, actuals); r2 = r2_score(actuals, preds)
        boot = []
        for _ in range(n_boot):
            bi = rng.choice(len(preds), len(preds), replace=True)
            if len(np.unique(actuals[bi])) > 2:
                boot.append(spearmanr(preds[bi], actuals[bi])[0])
        lo, hi = (np.percentile(boot, [2.5, 97.5]) if boot else (np.nan, np.nan))
        results[name] = {
            "spearman_pred_vs_actual": round(float(sr),4), "p": round(float(sp),4),
            "spearman_CI": (round(float(lo),4), round(float(hi),4)),
            "r2": round(float(r2),4), "n_pred": len(preds), "n_features": len(cols),
        }
    return results
'''
open(f"{REPO}/src/predict.py","w").write(SRC)
print("wrote src/predict.py:", len(SRC), "chars")
assert "assemble_features" in SRC and "evaluate_predictor" in SRC
print("OK — predict.py written")

wrote src/predict.py: 6024 chars
OK — predict.py written


In [ ]:
import importlib.util, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
for name in ["src.explain","src.predict"]:
    fn = name.split(".")[1]
    spec = importlib.util.spec_from_file_location(name, f"{REPO}/src/{fn}.py")
    mod = importlib.util.module_from_spec(spec); sys.modules[name] = mod; spec.loader.exec_module(mod)
import src.predict as predict
import importlib
from src import train
importlib.reload(train)
import pandas as pd, numpy as np

# M0 anchor + baseline features (from M0 ALONE — the whole point)
cnn, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
fdf = predict.assemble_features(cnn, df, splits, scaler, feat_cols, le)
print("baseline features assembled for", len(fdf), "classes\n")
print(fdf.round(3).to_string())

# build prediction table: (class, cell) -> features + actual Δrecall
delta = pd.read_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"), index_col=0)
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"].to_dict()
tbl = predict.build_prediction_table(fdf, delta, cells=("prune50","prune80","distillation"), tiers=tiers)
print(f"\nprediction table: {len(tbl)} (class,cell) rows across {tbl['class'].nunique()} classes")

# 1) FEATURE SCREEN: which single baseline feature predicts Δrecall best?
feat_cols_all = ["support","baseline_recall","etf_cos_to_others","margin",
                 "sim_nearest_higher_freq","confusion_offdiag","pred_entropy"]
print("\n" + "="*70 + "\nFEATURE SCREEN (rank-corr of each baseline feature vs actual Δrecall):")
print(predict.screen_features(tbl, feat_cols_all).to_string(index=False))

# 2) PREDICTOR COMPARISON (LOCO-CV) — Option 3: do confusability features beat the baselines?
sets = {
    "frequency_only (baseline a)": ["support"],
    "margin_only Tran-Fioretto (baseline b)": ["margin"],
    "neural_collapse (plan theory)": ["etf_cos_to_others","margin"],
    "confusability (Stage2 mechanism)": ["sim_nearest_higher_freq","confusion_offdiag","pred_entropy"],
    "confusability+freq": ["sim_nearest_higher_freq","confusion_offdiag","support"],
    "all_features": feat_cols_all,
}
print("\n" + "="*70 + "\nPREDICTOR COMPARISON (leave-one-class-out CV, Spearman pred-vs-actual):")
res = predict.evaluate_predictor(tbl, sets, n_boot=1000)
for k, v in res.items():
    if "spearman_pred_vs_actual" in v:
        print(f"  {k}: Spearman={v['spearman_pred_vs_actual']} CI={v['spearman_CI']} R2={v['r2']} (n={v['n_pred']})")
    else:
        print(f"  {k}: {v}")

fdf.to_csv(PATHS.tables("explain","cnn_predict_features.csv"))
tbl.to_csv(PATHS.tables("explain","cnn_predict_table.csv"), index=False)
print("\nsaved -> results/tables/explain/")

baseline features assembled for 34 classes

                         support  baseline_recall  etf_cos_to_others  margin  sim_nearest_higher_freq  confusion_offdiag  pred_entropy
Backdoor_Malware            70.0            0.000              0.199  -2.287                    0.947              1.000         2.563
BenignTraffic             3306.0            0.672              0.184   0.410                    0.865              0.328         2.219
BrowserHijacking           100.0            0.010              0.156  -2.607                    0.935              0.990         2.454
CommandInjection            95.0            0.063              0.160  -2.013                    0.808              0.937         2.201
DDoS-ACK_Fragmentation    3230.0            0.970             -0.120   6.339                    0.556              0.030         0.362
DDoS-HTTP_Flood            459.0            0.793             -0.009   1.533                    0.913              0.207         0.712
DDoS-ICMP_F

In [ ]:
import pandas as pd, numpy as np
import importlib.util, sys
spec = importlib.util.spec_from_file_location("src.predict", f"{REPO}/src/predict.py")
predict = importlib.util.module_from_spec(spec); sys.modules["src.predict"]=predict; spec.loader.exec_module(predict)

fdf = pd.read_csv(PATHS.tables("explain","cnn_predict_features.csv"), index_col=0)
delta = pd.read_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"), index_col=0)
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]

# --- per-CELL feature screen (NOT pooled): does any feature predict prune80 alone? ---
for cell in ["prune80","prune50","distillation"]:
    tbl1 = predict.build_prediction_table(fdf, delta, cells=(cell,), tiers=tiers.to_dict())
    print(f"\n=== {cell}: feature screen (per-class, n={len(tbl1)}) ===")
    print(predict.screen_features(tbl1, ["support","margin","etf_cos_to_others",
          "sim_nearest_higher_freq","confusion_offdiag","pred_entropy"]).to_string(index=False))

# --- the sharpest test: on the MEASURABLE tier only, prune80 only ---
# (the 13 classes where collapse is cleanly measured; this is where Stage-2 confusability lived)
meas = tiers[tiers=="measurable"].index
sub = fdf.loc[[c for c in meas if c in fdf.index]].copy()
sub["prune80_delta"] = delta.loc[sub.index,"prune80"]
from scipy.stats import spearmanr
print("\n=== MEASURABLE tier (n=13), prune80 only: feature vs Δrecall ===")
for f in ["support","margin","etf_cos_to_others","sim_nearest_higher_freq","confusion_offdiag","pred_entropy","baseline_recall"]:
    x = sub[f].astype(float); d = sub["prune80_delta"].astype(float)
    ok = x.notna()&d.notna()
    if ok.sum()>=5:
        r,p = spearmanr(x[ok],d[ok])
        print(f"  {f:26} Spearman={r:+.3f}  p={p:.3f}")


=== prune80: feature screen (per-class, n=34) ===
                feature  spearman  spearman_p  kendall  n
           pred_entropy   -0.0860      0.6286   0.0053 34
                 margin    0.0698      0.6948  -0.0125 34
                support   -0.0545      0.7593  -0.0303 34
      confusion_offdiag   -0.0460      0.7961   0.0643 34
sim_nearest_higher_freq    0.0344      0.8469   0.0089 34
      etf_cos_to_others    0.0280      0.8753   0.0374 34

=== prune50: feature screen (per-class, n=34) ===
                feature  spearman  spearman_p  kendall  n
sim_nearest_higher_freq   -0.4871      0.0035  -0.3283 34
                support    0.1710      0.3336   0.0785 34
      etf_cos_to_others    0.1018      0.5669   0.0642 34
                 margin    0.0973      0.5840   0.0464 34
      confusion_offdiag   -0.0864      0.6272  -0.0358 34
           pred_entropy   -0.0478      0.7882  -0.0285 34

=== distillation: feature screen (per-class, n=34) ===
                feature  spear

In [ ]:
import pandas as pd, numpy as np
from scipy.stats import spearmanr
fdf = pd.read_csv(PATHS.tables("explain","cnn_predict_features.csv"), index_col=0)
delta = pd.read_csv(PATHS.tables("compression","cnn1d_delta_recall.csv"), index_col=0)

# bootstrap CI on the prune50 confusability signal (the one significant result)
x = fdf["sim_nearest_higher_freq"].astype(float)
common = x.index.intersection(delta.index)
x = x.loc[common]; d50 = delta.loc[common,"prune50"].astype(float); d80 = delta.loc[common,"prune80"].astype(float)
ok = x.notna()&d50.notna()
rng = np.random.default_rng(0)
def boot_spearman(xx, dd, n=2000):
    xx, dd = xx.values, dd.values
    out=[]
    for _ in range(n):
        bi = rng.choice(len(xx), len(xx), replace=True)
        if len(np.unique(xx[bi]))>2: out.append(spearmanr(xx[bi],dd[bi])[0])
    return np.percentile(out,[2.5,50,97.5])

print("prune50 sim_nearest_higher_freq vs Δrecall:")
print(f"  point Spearman: {spearmanr(x[ok],d50[ok])[0]:+.3f}")
lo,med,hi = boot_spearman(x[ok],d50[ok])
print(f"  bootstrap 95% CI: [{lo:+.3f}, {hi:+.3f}]  median {med:+.3f}")
print(f"  -> CI excludes zero: {hi < 0}")

print("\nprune80 same feature (the null):")
print(f"  point Spearman: {spearmanr(x[ok],d80[ok])[0]:+.3f}")
lo,med,hi = boot_spearman(x[ok],d80[ok])
print(f"  bootstrap 95% CI: [{lo:+.3f}, {hi:+.3f}]")

# the gradient: confusability-vs-collapse correlation across compression severity
print("\nPREDICTABILITY-vs-SEVERITY gradient (does confusability's predictive power fade as compression bites harder?):")
for cell in ["prune50","distillation","prune80"]:
    dd = delta.loc[common,cell].astype(float); ok2=x.notna()&dd.notna()
    r,p = spearmanr(x[ok2],dd[ok2])
    mean_collapse = dd[dd<0].mean()
    print(f"  {cell:14} confusability-Spearman={r:+.3f} (p={p:.3f}) | mean Δrecall of collapsed={mean_collapse:+.3f}")

prune50 sim_nearest_higher_freq vs Δrecall:
  point Spearman: -0.487
  bootstrap 95% CI: [-0.717, -0.138]  median -0.491
  -> CI excludes zero: True

prune80 same feature (the null):
  point Spearman: +0.034
  bootstrap 95% CI: [-0.329, +0.387]

PREDICTABILITY-vs-SEVERITY gradient (does confusability's predictive power fade as compression bites harder?):
  prune50        confusability-Spearman=-0.487 (p=0.003) | mean Δrecall of collapsed=-0.070
  distillation   confusability-Spearman=+0.079 (p=0.656) | mean Δrecall of collapsed=-0.054
  prune80        confusability-Spearman=+0.034 (p=0.847) | mean Δrecall of collapsed=-0.221


In [ ]:
import pandas as pd, json, os
predict_result = {
    "stage": "PREDICT (Stage 3)",
    "verdict": "NEGATIVE / BOUNDED — per-class collapse not reliably forecastable from baseline features",
    "evidence": {
        "prune80_best_feature_spearman": 0.07,   # all features ns, |rho|<0.09
        "prune80_all_features_ns": True,
        "distillation_best_feature_spearman": -0.27,  # ns (p=0.13)
        "prune50_confusability_spearman": -0.487,
        "prune50_confusability_CI": [-0.717, -0.138],
        "prune50_confusability_significant": True,
        "gradient_monotone_with_severity": False,  # distillation milder than prune50 but NOT predictable -> no clean gradient
    },
    "neural_collapse_features_predict": False,   # etf_cos_to_others ~0 everywhere -> plan v2.1 NC-predictor premise unsupported
    "conclusion": "Baseline properties do not determine which classes an aggressively-compressed IDS abandons; "
                  "confusability predicts collapse only under moderate pruning (prune50), not under aggressive "
                  "pruning (prune80) or distillation. Collapse is a compression-model interaction, not forecastable "
                  "pre-compression. Per §9 fallback: PREDICT reported as bounded negative; paper led by EXPLAIN mechanism.",
    "paper_impact": "Headline shifts from predictive diagnostic to resolved mechanism + honest prediction-boundary; "
                    "venue leans Computers & Security; MITIGATE (decision-layer recovery, near-free per crux) becomes the practical payoff."
}
with open(f"{REPO}/results/tables/explain/predict_stage3_result.json","w") as f:
    json.dump(predict_result, f, indent=2)
print(json.dumps(predict_result, indent=2))

{
  "stage": "PREDICT (Stage 3)",
  "verdict": "NEGATIVE / BOUNDED \u2014 per-class collapse not reliably forecastable from baseline features",
  "evidence": {
    "prune80_best_feature_spearman": 0.07,
    "prune80_all_features_ns": true,
    "distillation_best_feature_spearman": -0.27,
    "prune50_confusability_spearman": -0.487,
    "prune50_confusability_CI": [
      -0.717,
      -0.138
    ],
    "prune50_confusability_significant": true,
    "gradient_monotone_with_severity": false
  },
  "neural_collapse_features_predict": false,
  "conclusion": "Baseline properties do not determine which classes an aggressively-compressed IDS abandons; confusability predicts collapse only under moderate pruning (prune50), not under aggressive pruning (prune80) or distillation. Collapse is a compression-model interaction, not forecastable pre-compression. Per \u00a79 fallback: PREDICT reported as bounded negative; paper led by EXPLAIN mechanism.",
  "paper_impact": "Headline shifts from predic

In [ ]:
import subprocess, os
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO)
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "PREDICT (Stage 3): bounded NEGATIVE per plan S9 fallback. Per-class collapse not forecastable from baseline features under aggressive pruning/distillation (all features ns, |rho|<0.09 on prune80); neural-collapse-predictor premise (plan v2.1) unsupported (etf~0). Sole signal: confusability under prune50 (rho=-0.49, CI[-0.72,-0.14]); does NOT generalize across methods. Paper reframes: EXPLAIN mechanism as spine + honest prediction-boundary; venue -> C&S; MITIGATE (decision-layer, near-free per crux) as practical payoff"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   d32669f..b977201  main -> main



In [ ]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/mitigate.py — Stage 4: predictor-FREE, measurement-guided decision-layer recovery.
The crux showed information SURVIVES under compression, so protection is DECISION-LAYER.
Recover prune80-collapsed classes by adjusting the decision rule on the FROZEN compressed
model. Temperature = negative control (can't move argmax). Per-class bias = the cheap fix.
ALL params fit on VAL, evaluated on TEST. Report the trade-off frontier, not a point.
"""
from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from .config import CFG, PATHS
from . import train as _train


def _logits_for_split(model, df, splits, scaler, feat_cols, le, which):
    import torch
    sub = df.loc[splits[which]]
    X = scaler.transform(sub[feat_cols].to_numpy(np.float32))
    y = le.transform(sub["label"].to_numpy())
    logits = _train._forward_batched(model, torch.tensor(X, dtype=torch.float32)).numpy()
    return logits, y


def _per_class_recall(logits, y, bias=None):
    pred = (logits + bias).argmax(1) if bias is not None else logits.argmax(1)
    classes = np.unique(y)
    return {int(c): float((pred[y == c] == c).mean()) for c in classes}, pred


def temperature_control(logits_te, y_te, T_grid=(0.5, 1.0, 2.0, 5.0)):
    rows = []
    for T in T_grid:
        pred = (logits_te / T).argmax(1)
        rows.append({"T": T, "macroF1": round(f1_score(y_te, pred, average="macro"), 4),
                     "accuracy": round(float((pred == y_te).mean()), 4)})
    return pd.DataFrame(rows)


def fit_per_class_bias(logits_val, y_val, target_classes, strength,
                       n_classes, max_bias=20.0, step=0.25):
    bias = np.zeros(n_classes)
    for c in target_classes:
        best_b, best_r = 0.0, _per_class_recall(logits_val, y_val, bias)[0].get(c, 0.0)
        b = 0.0
        while b < max_bias * strength:
            b += step
            trial = bias.copy(); trial[c] = b
            r = _per_class_recall(logits_val, y_val, trial)[0].get(c, 0.0)
            if r > best_r + 1e-4:
                best_r, best_b = r, b
            elif r < best_r - 0.02:
                break
        bias[c] = best_b
    return bias


def recovery_metrics(logits, y, bias, target_classes, le, benign_idx=None):
    base_rec, base_pred = _per_class_recall(logits, y, None)
    new_rec, new_pred = _per_class_recall(logits, y, bias)
    base_f1 = f1_score(y, base_pred, average="macro")
    new_f1 = f1_score(y, new_pred, average="macro")
    fpr_change = np.nan
    if benign_idx is not None:
        bm = (y == benign_idx)
        if bm.any():
            base_benign_recall = float((base_pred[bm] == benign_idx).mean())
            new_benign_recall = float((new_pred[bm] == benign_idx).mean())
            fpr_change = round(base_benign_recall - new_benign_recall, 4)
    target_gain = {le.classes_[c]: round(new_rec[c] - base_rec[c], 4) for c in target_classes}
    others = [c for c in base_rec if c not in target_classes]
    other_cost = {le.classes_[c]: round(new_rec[c] - base_rec[c], 4)
                  for c in others if abs(new_rec[c] - base_rec[c]) > 0.01}
    return {
        "macroF1_base": round(base_f1, 4), "macroF1_new": round(new_f1, 4),
        "macroF1_delta": round(new_f1 - base_f1, 4),
        "mean_target_recovery": round(np.mean(list(target_gain.values())), 4) if target_gain else 0.0,
        "benign_recall_drop_as_FPR": fpr_change,
        "target_gain": target_gain, "other_class_cost": other_cost,
    }


def frontier(model_comp, df, splits, scaler, feat_cols, le, target_classes,
             strengths=(0.0, 0.25, 0.5, 0.75, 1.0), benign_name="BenignTraffic"):
    Lval, yval = _logits_for_split(model_comp, df, splits, scaler, feat_cols, le, "val")
    Lte, yte = _logits_for_split(model_comp, df, splits, scaler, feat_cols, le, "test")
    n_classes = Lte.shape[1]
    tgt_idx = [int(np.where(le.classes_ == c)[0][0]) for c in target_classes if c in le.classes_]
    benign_idx = int(np.where(le.classes_ == benign_name)[0][0]) if benign_name in le.classes_ else None
    rows = []
    for s in strengths:
        bias = fit_per_class_bias(Lval, yval, tgt_idx, s, n_classes) if s > 0 else np.zeros(n_classes)
        m = recovery_metrics(Lte, yte, bias, tgt_idx, le, benign_idx)
        rows.append({"strength": s, "mean_target_recovery": m["mean_target_recovery"],
                     "macroF1_test": m["macroF1_new"], "macroF1_delta": m["macroF1_delta"],
                     "benign_FPR_proxy": m["benign_recall_drop_as_FPR"],
                     "n_other_classes_hurt": len(m["other_class_cost"])})
    return pd.DataFrame(rows), (Lval, yval, Lte, yte, tgt_idx, benign_idx)
'''
open(f"{REPO}/src/mitigate.py","w").write(SRC)
print("wrote src/mitigate.py:", len(SRC), "chars")
assert "frontier" in SRC and "temperature_control" in SRC
print("OK — mitigate.py written")

wrote src/mitigate.py: 4675 chars
OK — mitigate.py written


In [ ]:
import importlib.util, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
spec = importlib.util.spec_from_file_location("src.mitigate", f"{REPO}/src/mitigate.py")
mit = importlib.util.module_from_spec(spec); sys.modules["src.mitigate"]=mit; spec.loader.exec_module(mit)
import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd, numpy as np

# rebuild the prune80 CNN (the worst collapse) + identify the collapsed classes
cnn, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
p80, _, _ = compression.prune_and_finetune(cnn, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
flags = pd.read_csv(PATHS.tables("compression","cnn1d_collapse_flags.csv"), index_col=0)
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]
# target the measurable-tier classes that collapsed under prune80
collapsed = [c for c in flags.index[flags["prune80"]] if tiers.get(c)=="measurable"]
print("prune80 measurable collapsed classes to recover:", collapsed, "\n")

# 1) NEGATIVE CONTROL: temperature scaling can't move recall
Lte, yte = mit._logits_for_split(p80, df, splits, scaler, feat_cols, le, "test")
print("="*70 + "\nNEGATIVE CONTROL — scalar temperature (macroF1 MUST be flat: can't move argmax):")
print(mit.temperature_control(Lte, yte).to_string(index=False))

# 2) THE FIX: per-class bias, trade-off frontier (fit on VAL, eval on TEST)
print("\n" + "="*70 + "\nTRADE-OFF FRONTIER — per-class logit bias (fit VAL, eval TEST):")
fr, ctx = mit.frontier(p80, df, splits, scaler, feat_cols, le, collapsed)
print(fr.to_string(index=False))

# 3) detailed accounting at the best operating point (strength=1.0)
Lval, yval, Lte2, yte2, tgt_idx, benign_idx = ctx
bias = mit.fit_per_class_bias(Lval, yval, tgt_idx, 1.0, Lte2.shape[1])
detail = mit.recovery_metrics(Lte2, yte2, bias, tgt_idx, le, benign_idx)
print("\n" + "="*70 + "\nDETAIL at full strength — per-class recovery and cost:")
print("  macroF1:", detail["macroF1_base"], "->", detail["macroF1_new"], f"({detail['macroF1_delta']:+.4f})")
print("  mean target recovery:", detail["mean_target_recovery"])
print("  benign recall drop (FPR proxy):", detail["benign_recall_drop_as_FPR"])
print("\n  per-target-class recovery (Δrecall on test):")
for k,v in detail["target_gain"].items(): print(f"    {k}: {v:+.3f}")
print("\n  collateral cost to other classes (only those that moved):")
for k,v in detail["other_class_cost"].items(): print(f"    {k}: {v:+.3f}")

fr.to_csv(PATHS.tables("explain","mitigate_frontier_prune80.csv"), index=False)
print("\nsaved -> results/tables/explain/mitigate_frontier_prune80.csv")

prune80 measurable collapsed classes to recover: ['BenignTraffic', 'DNS_Spoofing', 'DictionaryBruteForce', 'DoS-HTTP_Flood', 'DoS-SYN_Flood', 'DoS-UDP_Flood', 'MITM-ArpSpoofing', 'Recon-HostDiscovery', 'Recon-OSScan', 'Recon-PortScan'] 

NEGATIVE CONTROL — scalar temperature (macroF1 MUST be flat: can't move argmax):
  T  macroF1  accuracy
0.5   0.3377    0.5524
1.0   0.3377    0.5524
2.0   0.3377    0.5524
5.0   0.3377    0.5524

TRADE-OFF FRONTIER — per-class logit bias (fit VAL, eval TEST):
 strength  mean_target_recovery  macroF1_test  macroF1_delta  benign_FPR_proxy  n_other_classes_hurt
     0.00                0.0000        0.3377         0.0000            0.0000                     0
     0.25                0.2605        0.3735         0.0358           -0.2796                     6
     0.50                0.3976        0.2803        -0.0574           -0.2796                    12
     0.75                0.3711        0.1276        -0.2101            0.1764                   

In [2]:
from google.colab import drive
drive.mount('/content/drive')
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
print("mitigate.py exists:", os.path.exists(f"{REPO}/src/mitigate.py"))

Mounted at /content/drive
mitigate.py exists: True


In [3]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
ADD = r'''

def fit_joint_bias_macroF1(logits_val, y_val, n_classes, *, n_restarts=3, iters=400, seed=0):
    """Coordinate-ascent per-class bias maximizing VAL macro-F1 (not greedy per-target)."""
    from sklearn.metrics import f1_score
    rng = np.random.default_rng(seed)
    classes = np.unique(y_val)
    def macro(bias):
        return f1_score(y_val, (logits_val + bias).argmax(1), average="macro")
    best_bias, best_f1 = np.zeros(n_classes), macro(np.zeros(n_classes))
    for r in range(n_restarts):
        bias = (rng.normal(0, 0.5, n_classes) if r > 0 else np.zeros(n_classes))
        cur = macro(bias)
        for it in range(iters):
            c = int(rng.choice(classes)); improved = False
            for delta in (0.5, -0.5, 0.25, -0.25, 1.0, -1.0):
                trial = bias.copy(); trial[c] += delta
                f = macro(trial)
                if f > cur + 1e-5:
                    bias, cur = trial, f; improved = True; break
        if cur > best_f1:
            best_bias, best_f1 = bias.copy(), cur
    return best_bias, best_f1


def refit_head(model_comp, df, splits, scaler, feat_cols, le, *, epochs=15, lr=1e-2,
               batch_size=4096, seed=0):
    """Re-fit ONLY a fresh linear head on the FROZEN compressed representation."""
    import torch, torch.nn as nn
    from . import explain as _explain
    from torch.utils.data import DataLoader, TensorDataset
    _train.set_all_seeds(seed)
    def feats(which):
        return _explain.extract_features(model_comp, df, splits, scaler, feat_cols, le, which=which)
    ftr, _, ytr = feats("train"); fva, _, yva = feats("val"); fte, _, yte = feats("test")
    d = ftr.shape[1]; K = len(le.classes_)
    head = nn.Linear(d, K).to(_train.DEVICE)
    w = _train.tempered_class_weights(ytr, K)
    crit = nn.CrossEntropyLoss(weight=w); opt = torch.optim.Adam(head.parameters(), lr=lr)
    Xtr = torch.tensor(ftr, dtype=torch.float32); Ytr = torch.tensor(ytr, dtype=torch.long)
    loader = DataLoader(TensorDataset(Xtr, Ytr), batch_size=batch_size, shuffle=True)
    for ep in range(epochs):
        head.train()
        for xb, yb in loader:
            xb, yb = xb.to(_train.DEVICE), yb.to(_train.DEVICE)
            opt.zero_grad(); crit(head(xb), yb).backward(); opt.step()
    head.eval()
    with torch.no_grad():
        Lva = head(torch.tensor(fva, dtype=torch.float32).to(_train.DEVICE)).cpu().numpy()
        Lte = head(torch.tensor(fte, dtype=torch.float32).to(_train.DEVICE)).cpu().numpy()
    return Lva, yva, Lte, yte


def pairwise_vs_ovr_separability(model_comp, model_M0, df, splits, scaler, feat_cols, le,
                                 class_pairs, which="test", max_n=40000, seed=0):
    """OvR probe AUC vs PAIRWISE (collapsed-vs-absorber) probe AUC, M0 vs compressed.
    OvR survives + pairwise collapses => explains zero-sum mitigation."""
    import numpy as np, pandas as pd
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    from sklearn.model_selection import train_test_split
    from . import explain as _explain
    rng = np.random.default_rng(seed)
    fM0, _, y = _explain.extract_features(model_M0, df, splits, scaler, feat_cols, le, which=which)
    fC, _, _ = _explain.extract_features(model_comp, df, splits, scaler, feat_cols, le, which=which)
    if len(y) > max_n:
        idx = rng.choice(len(y), max_n, replace=False); fM0, fC, y = fM0[idx], fC[idx], y[idx]
    name2i = {n: i for i, n in enumerate(le.classes_)}
    def ovr_auc(F, c):
        yb = (y == c).astype(int)
        if yb.sum() < 5: return np.nan
        tr, te = train_test_split(np.arange(len(y)), test_size=0.4, random_state=seed, stratify=yb)
        clf = LogisticRegression(max_iter=500, class_weight="balanced").fit(F[tr], yb[tr])
        return roc_auc_score(yb[te], clf.predict_proba(F[te])[:, 1])
    def pair_auc(F, c, a):
        m = (y == c) | (y == a)
        if (y == c).sum() < 5 or (y == a).sum() < 5: return np.nan
        Fm, ym = F[m], (y[m] == c).astype(int)
        tr, te = train_test_split(np.arange(len(ym)), test_size=0.4, random_state=seed, stratify=ym)
        clf = LogisticRegression(max_iter=500, class_weight="balanced").fit(Fm[tr], ym[tr])
        return roc_auc_score(ym[te], clf.predict_proba(Fm[te])[:, 1])
    rows = []
    for collapsed, absorber in class_pairs:
        c, a = name2i[collapsed], name2i[absorber]
        rows.append({"collapsed": collapsed, "absorber": absorber,
            "ovr_auc_M0": round(ovr_auc(fM0, c), 4), "ovr_auc_comp": round(ovr_auc(fC, c), 4),
            "pair_auc_M0": round(pair_auc(fM0, c, a), 4), "pair_auc_comp": round(pair_auc(fC, c, a), 4)})
    out = pd.DataFrame(rows)
    out["ovr_drop"] = (out["ovr_auc_M0"] - out["ovr_auc_comp"]).round(4)
    out["pair_drop"] = (out["pair_auc_M0"] - out["pair_auc_comp"]).round(4)
    return out
'''
with open(f"{REPO}/src/mitigate.py","a") as f:
    f.write(ADD)
print("appended B+C to mitigate.py")

appended B+C to mitigate.py


In [4]:
print("CFG/PATHS:", 'CFG' in dir() and 'PATHS' in dir(), "| df/splits:", 'df' in dir() and 'splits' in dir(),
      "| rows:", len(df) if 'df' in dir() else "—")

CFG/PATHS: False | df/splits: False | rows: —


In [5]:
# config + data reload (Drive already mounted since the append worked)
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
os.chdir(REPO); sys.path.insert(0, REPO)
from src.config import CFG, PATHS, set_all_seeds
import torch, pandas as pd, numpy as np
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

from src import data
df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("rows:", len(df))

GPU: Tesla T4
rows: 3661696


In [6]:
import importlib.util, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
spec = importlib.util.spec_from_file_location("src.mitigate", f"{REPO}/src/mitigate.py")
mit = importlib.util.module_from_spec(spec); sys.modules["src.mitigate"]=mit; spec.loader.exec_module(mit)
import importlib
from src import train, compression
for m in (train, compression): importlib.reload(m)
import pandas as pd, numpy as np
from sklearn.metrics import f1_score

cnn, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
p80, _, _ = compression.prune_and_finetune(cnn, df, "ciciot2023", splits, CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
print("prune80 ready\n")

Lval, yval = mit._logits_for_split(p80, df, splits, scaler, feat_cols, le, "val")
Lte, yte = mit._logits_for_split(p80, df, splits, scaler, feat_cols, le, "test")
base_f1 = f1_score(yte, Lte.argmax(1), average="macro")
print(f"prune80 baseline test macroF1: {base_f1:.4f}\n")

print("="*70 + "\nB1 — JOINT macroF1 bias (proper optimizer, not greedy):")
bias, valf1 = mit.fit_joint_bias_macroF1(Lval, yval, Lte.shape[1], iters=600)
b1_f1 = f1_score(yte, (Lte+bias).argmax(1), average="macro")
print(f"  test macroF1: {base_f1:.4f} -> {b1_f1:.4f} ({b1_f1-base_f1:+.4f})   [val-opt {valf1:.4f}]")

print("\n" + "="*70 + "\nB2 — RE-FIT HEAD on frozen compressed representation:")
Lva2, yva2, Lte2, yte2 = mit.refit_head(p80, df, splits, scaler, feat_cols, le, epochs=15)
b2_f1 = f1_score(yte2, Lte2.argmax(1), average="macro")
print(f"  test macroF1: {base_f1:.4f} -> {b2_f1:.4f} ({b2_f1-base_f1:+.4f})")
flags = pd.read_csv(PATHS.tables("compression","cnn1d_collapse_flags.csv"), index_col=0)
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]
collapsed = [c for c in flags.index[flags["prune80"]] if tiers.get(c)=="measurable"]
base_rec = {le.classes_[c]: float((Lte.argmax(1)[yte==c]==c).mean()) for c in np.unique(yte)}
new_rec = {le.classes_[c]: float((Lte2.argmax(1)[yte2==c]==c).mean()) for c in np.unique(yte2)}
print("  per-collapsed-class recall, prune80 -> refit-head:")
for c in collapsed:
    print(f"    {c}: {base_rec[c]:.3f} -> {new_rec[c]:.3f} ({new_rec[c]-base_rec[c]:+.3f})")

print("\n" + "="*70 + "\nC — PAIRWISE vs ONE-VS-REST separability:")
pairs = [("DoS-UDP_Flood","DDoS-UDP_Flood"), ("DoS-SYN_Flood","DDoS-SYN_Flood"),
         ("DoS-HTTP_Flood","DDoS-HTTP_Flood"), ("Recon-PortScan","VulnerabilityScan"),
         ("Recon-OSScan","VulnerabilityScan"), ("Recon-HostDiscovery","VulnerabilityScan")]
pc = mit.pairwise_vs_ovr_separability(p80, cnn, df, splits, scaler, feat_cols, le, pairs)
print(pc.to_string(index=False))

print("\n" + "="*70 + "\nSUMMARY:")
print(f"  greedy bias (prior run): macroF1 -> 0.05 (destructive)")
print(f"  B1 joint bias:           macroF1 -> {b1_f1:.4f}")
print(f"  B2 refit head:           macroF1 -> {b2_f1:.4f}")
print(f"  -> recovery {'IS possible' if max(b1_f1,b2_f1) > base_f1 else 'NOT possible'} without razing (vs base {base_f1:.4f})")

pc.to_csv(PATHS.tables("explain","pairwise_vs_ovr_prune80.csv"), index=False)
pd.DataFrame([{"method":"baseline","macroF1":base_f1},{"method":"joint_bias","macroF1":b1_f1},
              {"method":"refit_head","macroF1":b2_f1}]).to_csv(PATHS.tables("explain","mitigate_methods_prune80.csv"), index=False)
print("\nsaved -> results/tables/explain/")

prune80 ready

prune80 baseline test macroF1: 0.3377

B1 — JOINT macroF1 bias (proper optimizer, not greedy):
  test macroF1: 0.3377 -> 0.4428 (+0.1051)   [val-opt 0.4798]

B2 — RE-FIT HEAD on frozen compressed representation:
  test macroF1: 0.3377 -> 0.5257 (+0.1880)
  per-collapsed-class recall, prune80 -> refit-head:
    BenignTraffic: 0.179 -> 0.631 (+0.452)
    DNS_Spoofing: 0.462 -> 0.417 (-0.046)
    DictionaryBruteForce: 0.219 -> 0.452 (+0.233)
    DoS-HTTP_Flood: 0.213 -> 0.833 (+0.620)
    DoS-SYN_Flood: 0.000 -> 0.155 (+0.155)
    DoS-UDP_Flood: 0.000 -> 0.786 (+0.786)
    MITM-ArpSpoofing: 0.007 -> 0.133 (+0.126)
    Recon-HostDiscovery: 0.076 -> 0.648 (+0.572)
    Recon-OSScan: 0.000 -> 0.125 (+0.125)
    Recon-PortScan: 0.003 -> 0.118 (+0.116)

C — PAIRWISE vs ONE-VS-REST separability:


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


          collapsed          absorber  ovr_auc_M0  ovr_auc_comp  pair_auc_M0  pair_auc_comp  ovr_drop  pair_drop
      DoS-UDP_Flood    DDoS-UDP_Flood      0.9925        0.9924       0.7888         0.7862    0.0001     0.0026
      DoS-SYN_Flood    DDoS-SYN_Flood      0.9872        0.9862       0.8846         0.8833    0.0010     0.0013
     DoS-HTTP_Flood   DDoS-HTTP_Flood      0.9984        0.9978       0.9536         0.9402    0.0006     0.0134
     Recon-PortScan VulnerabilityScan      0.9315        0.9193       0.8223         0.7956    0.0122     0.0267
       Recon-OSScan VulnerabilityScan      0.9301        0.9201       0.8507         0.8095    0.0100     0.0412
Recon-HostDiscovery VulnerabilityScan      0.9830        0.9734       0.9397         0.9248    0.0096     0.0149

SUMMARY:
  greedy bias (prior run): macroF1 -> 0.05 (destructive)
  B1 joint bias:           macroF1 -> 0.4428
  B2 refit head:           macroF1 -> 0.5257
  -> recovery IS possible without razing (vs base 0.

In [7]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "MITIGATE: decision-layer recovery WORKS. Re-fit head on frozen prune80 representation: macroF1 0.338->0.526 (near uncompressed 0.545); collapsed classes recovered (DoS-UDP 0->0.79, DoS-HTTP 0.21->0.83, Recon-HostDisc 0.08->0.65). Greedy bias disaster was method artifact (joint optimizer +0.105). Pairwise diagnostic (C): BOTH ovr AND pairwise separability survive compression (pair_drop<0.04) -> collapse is purely decision-layer boundary misplacement, NOT representational loss; re-fit head re-places boundary. Residual hard classes (DoS-SYN, OSScan, PortScan, MITM) bound the blind spot. Mechanism->fix chain complete & self-consistent"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   b977201..7380fe1  main -> main



In [8]:
# bootstrap check + load TON
import os, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
if 'CFG' not in dir():
    os.chdir(REPO); sys.path.insert(0, REPO)
    from src.config import CFG, PATHS, set_all_seeds
    import torch, pandas as pd, numpy as np
import importlib
from src import data
importlib.reload(data)

# load + clean TON_IoT (strips identity fields, dedups)
df_ton = data.clean(data.load_raw("ton_iot"), "ton_iot")
print("TON rows:", len(df_ton))
print("TON columns:", [c for c in df_ton.columns if c not in ('label','group','_part','_row')][:20])
print("n features:", len([c for c in df_ton.columns if c not in ('label','group','_part','_row')]))
print("\nTON class distribution:")
print(df_ton['label'].value_counts().to_string())

TON rows: 0
TON columns: ['proto', 'service', 'duration', 'src_bytes', 'dst_bytes', 'conn_state', 'missed_bytes', 'src_pkts', 'src_ip_bytes', 'dst_pkts', 'dst_ip_bytes', 'dns_qclass', 'dns_qtype', 'dns_rcode', 'dns_AA', 'dns_RD', 'dns_RA', 'dns_rejected', 'ssl_version', 'ssl_cipher']
n features: 30

TON class distribution:
Series([], )


In [9]:
import sys, inspect
from src import data
print(inspect.getsource(data.clean))

def clean(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """Identity-field removal + dedup. MECHANISM-CRITICAL. Done on FEATURES only;
    label/group/order-helper columns are preserved."""
    keep_meta = [c for c in (LABEL_COL, GROUP_COL, "_part", "_row") if c in df.columns]
    drop = [c for c in IDENTITY_FIELDS.get(dataset, []) if c in df.columns]
    df = df.drop(columns=drop)
    # numeric coercion + drop rows with inf/NaN introduced by CICFlowMeter
    feat = [c for c in df.columns if c not in keep_meta]
    df[feat] = df[feat].apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=feat).reset_index(drop=True)
    # dedup on FEATURES + label (near-duplicate flow rows). Keep order helpers out of the key.
    dedup_key = [c for c in df.columns if c not in ("_part", "_row")]
    df = df.drop_duplicates(subset=dedup_key).reset_index(drop=True)
    return df



In [10]:
import importlib
from src import data
importlib.reload(data)
raw = data.load_raw("ton_iot")
print("raw TON rows:", len(raw))
print("\ndtypes:")
print(raw.dtypes.to_string())
print("\nNaN counts (top 15):")
print(raw.isna().sum().sort_values(ascending=False).head(15).to_string())
print("\ncategorical/placeholder check:")
for c in ['proto','service','conn_state','ssl_version','ssl_cipher','dns_query','http_method','weird_name']:
    if c in raw.columns:
        vals = raw[c].astype(str)
        print(f"  {c}: {(vals=='-').sum()} dashes, {vals.nunique()} unique, sample {list(vals.unique()[:5])}")

raw TON rows: 211043

dtypes:
src_ip                     object
src_port                    int64
dst_ip                     object
dst_port                    int64
proto                      object
service                    object
duration                  float64
src_bytes                   int64
dst_bytes                   int64
conn_state                 object
missed_bytes                int64
src_pkts                    int64
src_ip_bytes                int64
dst_pkts                    int64
dst_ip_bytes                int64
dns_query                  object
dns_qclass                  int64
dns_qtype                   int64
dns_rcode                   int64
dns_AA                     object
dns_RD                     object
dns_RA                     object
dns_rejected               object
ssl_version                object
ssl_cipher                 object
ssl_resumed                object
ssl_established            object
ssl_subject                object
ssl_issuer        

In [12]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
src = open(f"{REPO}/src/data.py").read()

# replace the clean() body with a dataset-aware version (CICIoT path unchanged; TON path added)
old = '''def clean(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """Identity-field removal + dedup. MECHANISM-CRITICAL. Done on FEATURES only;
    label/group/order-helper columns are preserved."""
    keep_meta = [c for c in (LABEL_COL, GROUP_COL, "_part", "_row") if c in df.columns]
    drop = [c for c in IDENTITY_FIELDS.get(dataset, []) if c in df.columns]
    df = df.drop(columns=drop)
    # numeric coercion + drop rows with inf/NaN introduced by CICFlowMeter
    feat = [c for c in df.columns if c not in keep_meta]
    df[feat] = df[feat].apply(pd.to_numeric, errors="coerce")
    df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=feat).reset_index(drop=True)
    # dedup on FEATURES + label (near-duplicate flow rows). Keep order helpers out of the key.
    dedup_key = [c for c in df.columns if c not in ("_part", "_row")]
    df = df.drop_duplicates(subset=dedup_key).reset_index(drop=True)
    return df'''

new = '''def clean(df: pd.DataFrame, dataset: str) -> pd.DataFrame:
    """Identity-field removal + dedup. MECHANISM-CRITICAL. Done on FEATURES only;
    label/group/order-helper columns are preserved. TON_IoT has mixed categorical+
    numeric features, so categoricals are label-encoded (structural '-' kept as a
    category) rather than coerced-to-NaN-and-dropped."""
    from pandas.api.types import is_numeric_dtype
    from sklearn.preprocessing import LabelEncoder
    keep_meta = [c for c in (LABEL_COL, GROUP_COL, "_part", "_row") if c in df.columns]
    drop = [c for c in IDENTITY_FIELDS.get(dataset, []) if c in df.columns]
    df = df.drop(columns=drop)
    feat = [c for c in df.columns if c not in keep_meta]

    # categorical features (non-numeric dtype): label-encode. '-' / absent becomes a
    # category, which is meaningful (a flow with no SSL differs structurally from one
    # with SSL). High-cardinality identity-like strings are already dropped via
    # IDENTITY_FIELDS. CICIoT2023 is all-numeric so this loop is a no-op there.
    cat = [c for c in feat if not is_numeric_dtype(df[c])]
    for c in cat:
        df[c] = LabelEncoder().fit_transform(df[c].astype(str)).astype(np.int64)

    # numeric coercion + drop rows with genuine inf/NaN in the numeric features only
    num = [c for c in feat if c not in cat]
    if num:
        df[num] = df[num].apply(pd.to_numeric, errors="coerce")
        df = df.replace([np.inf, -np.inf], np.nan).dropna(subset=num).reset_index(drop=True)

    # dedup on FEATURES + label (near-duplicate flow rows). Keep order helpers out of the key.
    dedup_key = [c for c in df.columns if c not in ("_part", "_row")]
    df = df.drop_duplicates(subset=dedup_key).reset_index(drop=True)
    return df'''

assert old in src, "clean() body not found verbatim — paste me the current clean() and I'll re-match"
src = src.replace(old, new)
open(f"{REPO}/src/data.py","w").write(src)
print("patched data.clean() with TON categorical handling")
print("CICIoT path unchanged (no-op for all-numeric); TON categoricals now label-encoded")

patched data.clean() with TON categorical handling
CICIoT path unchanged (no-op for all-numeric); TON categoricals now label-encoded


In [13]:
import importlib
from src import data
importlib.reload(data)
df_ton = data.clean(data.load_raw("ton_iot"), "ton_iot")
print("TON rows after fix:", len(df_ton))
n_feat = len([c for c in df_ton.columns if c not in ('label','group','_part','_row')])
print("n features:", n_feat, "| all numeric:", df_ton.drop(columns=['label','group']).select_dtypes(exclude='number').empty)
print("\nTON class distribution:")
print(df_ton['label'].value_counts().to_string())

TON rows after fix: 93644
n features: 30 | all numeric: True

TON class distribution:
label
normal        23639
injection     19849
password      15248
ddos          11962
xss            8740
dos            6448
scanning       4493
backdoor       1966
mitm           1039
ransomware      260


In [14]:
import importlib.util, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
for name in ["src.explain","src.mitigate"]:
    fn=name.split(".")[1]
    spec=importlib.util.spec_from_file_location(name, f"{REPO}/src/{fn}.py")
    mod=importlib.util.module_from_spec(spec); sys.modules[name]=mod; spec.loader.exec_module(mod)
import src.explain as explain, src.mitigate as mit
import importlib
from src import data, train, compression
for m in (data, train, compression): importlib.reload(m)
import pandas as pd, numpy as np
from sklearn.metrics import f1_score

# TON split (single capture -> row-order/stratified, per loader note)
splits_ton = data.temporal_within_capture_split(df_ton, seed=CFG["anchor_seed"])
print("TON splits:", {k: len(v) for k,v in splits_ton.items() if k in ('train','val','test')})

# === 1) TRAIN CNN anchor on TON ===
print("\n[1] training TON CNN anchor...")
ton_cnn, ton_info = train.train_model("cnn1d", df_ton, "ton_iot", splits_ton, seed=CFG["anchor_seed"],
                                      epochs=40, patience=6, batch_size=4096, arch_kwargs={"channels":(64,128)})
le = ton_info["label_encoder"]; scaler = ton_info["scaler"]; feat_cols = ton_info["feat_cols"]
print(f"  TON CNN: {ton_info['params']} params, test macroF1={ton_info['macroF1_test']:.4f}")
yt, yp, _ = train.predict(ton_cnn, df_ton, splits_ton, le, scaler, feat_cols)
r0 = train.per_class_recall_table(yt, yp, le).set_index("label")["recall"]
print("  baseline per-class recall:")
print(r0.round(3).to_string())

# === 2) PRUNE80 + measure collapse ===
print("\n[2] prune80 + per-class collapse...")
ton_p80, _, _ = compression.prune_and_finetune(ton_cnn, df_ton, "ton_iot", splits_ton, CFG["anchor_seed"], 0.80, arch="cnn1d", verbose=False)
ytp, ypp, _ = train.predict(ton_p80, df_ton, splits_ton, le, scaler, feat_cols)
r8 = train.per_class_recall_table(ytp, ypp, le).set_index("label")["recall"]
macro8 = f1_score(ytp, ypp, average="macro")
delta = (r8 - r0)
print(f"  prune80 macroF1: {ton_info['macroF1_test']:.4f} -> {macro8:.4f}")
print("  per-class recall M0 -> prune80 (Δ):")
for c in r0.index:
    flag = " <-- COLLAPSE" if delta[c] < -0.15 else ""
    print(f"    {c}: {r0[c]:.3f} -> {r8[c]:.3f} ({delta[c]:+.3f}){flag}")
collapsed = [c for c in delta.index if delta[c] < -0.15]
print(f"  collapsed (>0.15 drop): {collapsed}")

# === 3) CRUX: is it decision-layer or info-loss on TON? ===
print("\n[3] crux probe (M0 vs prune80)...")
crux = explain.crux_probe(ton_cnn, ton_p80, df_ton, splits_ton, scaler, feat_cols, le)
crux["delta_recall"] = delta
print(crux[["M0_probe_auc","comp_probe_auc","probe_auc_drop","delta_recall"]].round(3).to_string())
if collapsed:
    cset = crux.loc[[c for c in collapsed if c in crux.index]]
    print(f"  among collapsed: mean probe AUC drop {cset['probe_auc_drop'].mean():.4f}, "
          f"keeping AUC>0.85: {(cset['comp_probe_auc']>0.85).sum()}/{len(cset)}")

# === 4) MITIGATE: does re-fit head recover on TON? ===
print("\n[4] mitigate via re-fit head...")
Lte, yte = mit._logits_for_split(ton_p80, df_ton, splits_ton, scaler, feat_cols, le, "test")
base_f1 = f1_score(yte, Lte.argmax(1), average="macro")
Lva2, yva2, Lte2, yte2 = mit.refit_head(ton_p80, df_ton, splits_ton, scaler, feat_cols, le, epochs=15)
b2_f1 = f1_score(yte2, Lte2.argmax(1), average="macro")
print(f"  re-fit head: prune80 macroF1 {base_f1:.4f} -> {b2_f1:.4f} ({b2_f1-base_f1:+.4f})")
if collapsed:
    base_rec = {le.classes_[c]: float((Lte.argmax(1)[yte==c]==c).mean()) for c in np.unique(yte)}
    new_rec = {le.classes_[c]: float((Lte2.argmax(1)[yte2==c]==c).mean()) for c in np.unique(yte2)}
    print("  collapsed-class recovery:")
    for c in collapsed:
        print(f"    {c}: {base_rec[c]:.3f} -> {new_rec[c]:.3f} ({new_rec[c]-base_rec[c]:+.3f})")

print("\n" + "="*70 + "\nTON TRIO VERDICT:")
print(f"  collapse happens: {'YES' if collapsed else 'NO'} ({len(collapsed)} classes)")
if collapsed and 'cset' in dir():
    print(f"  decision-layer (info preserved): {'YES' if cset['probe_auc_drop'].mean()<0.05 else 'NO'}")
print(f"  re-fit head recovers: {'YES' if b2_f1>base_f1+0.02 else 'NO/MARGINAL'}")

# save
pd.DataFrame({"M0":r0,"prune80":r8,"delta":delta}).to_csv(PATHS.tables("compression","ton_cnn_prune80_recall.csv"))
crux.to_csv(PATHS.tables("explain","ton_crux_probe.csv"))
print("saved -> results/tables/")

TON splits: {'train': 65548, 'val': 14046, 'test': 14050}

[1] training TON CNN anchor...
  epoch 00  val_macroF1=0.0226
  epoch 01  val_macroF1=0.0629
  epoch 02  val_macroF1=0.2101
  epoch 03  val_macroF1=0.2201
  epoch 04  val_macroF1=0.3032
  epoch 05  val_macroF1=0.3553
  epoch 06  val_macroF1=0.4249
  epoch 07  val_macroF1=0.4174
  epoch 08  val_macroF1=0.4243
  epoch 09  val_macroF1=0.4359
  epoch 10  val_macroF1=0.4568
  epoch 11  val_macroF1=0.4661
  epoch 12  val_macroF1=0.4505
  epoch 13  val_macroF1=0.4598
  epoch 14  val_macroF1=0.5163
  epoch 15  val_macroF1=0.4777
  epoch 16  val_macroF1=0.4838
  epoch 17  val_macroF1=0.4742
  epoch 18  val_macroF1=0.5077
  epoch 19  val_macroF1=0.5440
  epoch 20  val_macroF1=0.5385
  epoch 21  val_macroF1=0.5445
  epoch 22  val_macroF1=0.5232
  epoch 23  val_macroF1=0.5500
  epoch 24  val_macroF1=0.4960
  epoch 25  val_macroF1=0.5203
  epoch 26  val_macroF1=0.5061
  epoch 27  val_macroF1=0.5561
  epoch 28  val_macroF1=0.4738
  epoch 29 

In [15]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "TON_IoT cross-dataset validation (headline trio): mechanism REPLICATES. CNN baseline macroF1 0.459; prune80 collapses ransomware (0.95->0) and scanning (0.67->0). CRUX: decision-layer confirmed (collapsed classes keep probe AUC, ransomware 0.999->0.999 despite recall 0.95->0). MITIGATE: re-fit head recovers (macroF1 0.288->0.447; ransomware 0->0.949 full recovery, scanning 0->0.764). Same decision-layer mechanism + same fix on 2nd dataset, milder imbalance, categorical features. Collapse extent scales with imbalance severity (TON focal vs CICIoT sink-cluster)"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   7380fe1..1e08886  main -> main



In [16]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
SRC = r'''
"""
src/ablation.py — D4: rarity-vs-separability ablation.
Subsample a class's TRAIN rows to a target (rare) count, retrain, prune80, measure if
it collapses. Run for a SEPARABLE class and a CONFUSABLE class at the SAME count:
  both collapse -> rarity sufficient; separable resists -> separability matters beyond rarity.
Test set never subsampled.
"""
from __future__ import annotations
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score

from .config import CFG, PATHS
from . import train as _train
from . import compression as _comp


def subsample_class_in_train(df, splits, class_name, target_count, seed=0):
    rng = np.random.default_rng(seed)
    train_idx = np.array(splits["train"])
    labels = df.loc[train_idx, "label"].to_numpy()
    keep_mask = labels != class_name
    cls_pos = np.where(labels == class_name)[0]
    if len(cls_pos) > target_count:
        chosen = rng.choice(cls_pos, target_count, replace=False)
        keep_mask[chosen] = True
    else:
        keep_mask[cls_pos] = True
    new_train = train_idx[keep_mask]
    return {**splits, "train": list(new_train)}


def run_ablation_for_class(df, splits, dataset, class_name, target_count, *,
                           arch="cnn1d", arch_kwargs=None, seed=0,
                           epochs=40, patience=6, ft_epochs=8):
    sp = subsample_class_in_train(df, splits, class_name, target_count, seed=seed)
    n_kept = int((df.loc[sp["train"], "label"] == class_name).sum())
    model, info = _train.train_model(arch, df, dataset, sp, seed, epochs=epochs,
                                     patience=patience, arch_kwargs=arch_kwargs,
                                     save=False, verbose=False)
    le, scaler, feat = info["label_encoder"], info["scaler"], info["feat_cols"]
    yt, yp, _ = _train.predict(model, df, sp, le, scaler, feat)
    r0 = _train.per_class_recall_table(yt, yp, le).set_index("label")["recall"]
    p80, _, _ = _comp.prune_and_finetune(model, df, dataset, sp, seed, 0.80,
                                         ft_epochs=ft_epochs, arch=arch, verbose=False)
    ytp, ypp, _ = _train.predict(p80, df, sp, le, scaler, feat)
    r8 = _train.per_class_recall_table(ytp, ypp, le).set_index("label")["recall"]
    return {
        "class": class_name, "kept_train_count": n_kept, "target_count": target_count,
        "macroF1_M0": round(info["macroF1_test"], 4),
        "macroF1_prune80": round(f1_score(ytp, ypp, average="macro"), 4),
        "class_recall_M0": round(float(r0.get(class_name, np.nan)), 4),
        "class_recall_prune80": round(float(r8.get(class_name, np.nan)), 4),
        "class_delta": round(float(r8.get(class_name, np.nan) - r0.get(class_name, np.nan)), 4),
    }
'''
open(f"{REPO}/src/ablation.py","w").write(SRC)
print("wrote src/ablation.py:", len(SRC), "chars")
print("OK")

wrote src/ablation.py: 2723 chars
OK


In [17]:
import importlib.util, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
spec = importlib.util.spec_from_file_location("src.ablation", f"{REPO}/src/ablation.py")
ab = importlib.util.module_from_spec(spec); sys.modules["src.ablation"]=ab; spec.loader.exec_module(ab)
import importlib
from src import data, train, compression
for m in (data, train, compression): importlib.reload(m)
import pandas as pd

# ensure we're on CICIoT (not TON). reload if df is TON or missing.
need_reload = ('df' not in dir()) or ('ransomware' in set(df['label'].unique()) if 'df' in dir() else True)
if need_reload:
    df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
    splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("CICIoT rows:", len(df), "| classes:", df['label'].nunique())

TARGET = 300   # subsample both classes' TRAIN to this rare count
print(f"\nD4: subsample each class's TRAIN to {TARGET}, retrain, prune80, measure collapse\n")

# SEPARABLE robust class (near-zero confusion at full freq)
print("[A] SEPARABLE class: DDoS-RSTFINFlood (offdiag~0.001, recall 0.999 at full freq)")
sep = ab.run_ablation_for_class(df, splits, "ciciot2023", "DDoS-RSTFINFlood", TARGET,
                                arch_kwargs={"channels":(64,128)})
print("   ", sep)

# CONFUSABLE class (high confusion, collapses at full freq)
print("\n[B] CONFUSABLE class: DoS-UDP_Flood (offdiag~0.32, collapsed at full freq)")
conf = ab.run_ablation_for_class(df, splits, "ciciot2023", "DoS-UDP_Flood", TARGET,
                                 arch_kwargs={"channels":(64,128)})
print("   ", conf)

print("\n" + "="*70 + "\nD4 VERDICT — rarity vs separability:")
print(f"  SEPARABLE  (RSTFINFlood) made rare to {TARGET}: recall {sep['class_recall_M0']} -> {sep['class_recall_prune80']} (Δ {sep['class_delta']})")
print(f"  CONFUSABLE (DoS-UDP)     made rare to {TARGET}: recall {conf['class_recall_M0']} -> {conf['class_recall_prune80']} (Δ {conf['class_delta']})")
print()
sep_collapsed = sep['class_delta'] < -0.15
conf_collapsed = conf['class_delta'] < -0.15
if conf_collapsed and not sep_collapsed:
    print("  -> SEPARABILITY MATTERS: separable class resists collapse even when rare;")
    print("     confusable class collapses. Collapse is driven by separability/confusability, NOT mere rarity.")
elif sep_collapsed and conf_collapsed:
    print("  -> RARITY SUFFICIENT: both collapse when made rare. Rarity alone drives collapse.")
elif not sep_collapsed and not conf_collapsed:
    print("  -> neither collapsed at this count; try a lower TARGET or check baseline recall held.")
else:
    print("  -> mixed/unexpected; inspect baseline recalls (did the separable class even learn at this count?)")

import json
with open(f"{REPO}/results/tables/explain/d4_rarity_vs_separability.json","w") as f:
    json.dump({"target_count":TARGET,"separable":sep,"confusable":conf}, f, indent=2)
print("\nsaved -> results/tables/explain/d4_rarity_vs_separability.json")

CICIoT rows: 3661696 | classes: 34

D4: subsample each class's TRAIN to 300, retrain, prune80, measure collapse

[A] SEPARABLE class: DDoS-RSTFINFlood (offdiag~0.001, recall 0.999 at full freq)
    {'class': 'DDoS-RSTFINFlood', 'kept_train_count': 300, 'target_count': 300, 'macroF1_M0': 0.5366, 'macroF1_prune80': 0.1378, 'class_recall_M0': 0.9986, 'class_recall_prune80': 0.6569, 'class_delta': -0.3418}

[B] CONFUSABLE class: DoS-UDP_Flood (offdiag~0.32, collapsed at full freq)
    {'class': 'DoS-UDP_Flood', 'kept_train_count': 300, 'target_count': 300, 'macroF1_M0': 0.5202, 'macroF1_prune80': 0.2789, 'class_recall_M0': 0.1297, 'class_recall_prune80': 0.0, 'class_delta': -0.1297}

D4 VERDICT — rarity vs separability:
  SEPARABLE  (RSTFINFlood) made rare to 300: recall 0.9986 -> 0.6569 (Δ -0.3418)
  CONFUSABLE (DoS-UDP)     made rare to 300: recall 0.1297 -> 0.0 (Δ -0.1297)

  -> mixed/unexpected; inspect baseline recalls (did the separable class even learn at this count?)

saved -> resu

In [18]:
import importlib.util, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
spec = importlib.util.spec_from_file_location("src.ablation", f"{REPO}/src/ablation.py")
ab = importlib.util.module_from_spec(spec); sys.modules["src.ablation"]=ab; spec.loader.exec_module(ab)
import pandas as pd

TARGET = 300
# separable (low offdiag, robust) vs confusable (high offdiag) — multiple of each
separable_classes  = ["DDoS-RSTFINFlood", "DDoS-ICMP_Flood", "Mirai-greip_flood"]
confusable_classes = ["DoS-UDP_Flood", "DoS-TCP_Flood", "DDoS-SynonymousIP_Flood"]

rows = []
for grp, classes in [("separable", separable_classes), ("confusable", confusable_classes)]:
    for cn in classes:
        r = ab.run_ablation_for_class(df, splits, "ciciot2023", cn, TARGET,
                                      arch_kwargs={"channels":(64,128)})
        r["group"] = grp
        rows.append(r)
        print(f"  [{grp}] {cn}: M0 recall (rare) {r['class_recall_M0']:.3f} -> prune80 {r['class_recall_prune80']:.3f}")

res = pd.DataFrame(rows)[["group","class","kept_train_count","class_recall_M0","class_recall_prune80","class_delta"]]
print("\n" + "="*70 + "\nD4 MULTI-CLASS (all subsampled to TARGET={}):".format(TARGET))
print(res.to_string(index=False))
print("\nMEAN baseline (M0) recall when made rare:")
print(f"  SEPARABLE classes:  {res[res.group=='separable']['class_recall_M0'].mean():.3f}")
print(f"  CONFUSABLE classes: {res[res.group=='confusable']['class_recall_M0'].mean():.3f}")
print("  -> if separable >> confusable at matched rare count, separability (not rarity) is the cause")
res.to_csv(PATHS.tables("explain","d4_multiclass.csv"), index=False)
print("\nsaved -> results/tables/explain/d4_multiclass.csv")

  [separable] DDoS-RSTFINFlood: M0 recall (rare) 0.999 -> prune80 0.657
  [separable] DDoS-ICMP_Flood: M0 recall (rare) 0.997 -> prune80 0.991
  [separable] Mirai-greip_flood: M0 recall (rare) 0.885 -> prune80 0.979
  [confusable] DoS-UDP_Flood: M0 recall (rare) 0.130 -> prune80 0.000
  [confusable] DoS-TCP_Flood: M0 recall (rare) 0.000 -> prune80 0.000
  [confusable] DDoS-SynonymousIP_Flood: M0 recall (rare) 0.000 -> prune80 0.000

D4 MULTI-CLASS (all subsampled to TARGET=300):
     group                   class  kept_train_count  class_recall_M0  class_recall_prune80  class_delta
 separable        DDoS-RSTFINFlood               300           0.9986                0.6569      -0.3418
 separable         DDoS-ICMP_Flood               300           0.9970                0.9911      -0.0059
 separable       Mirai-greip_flood               300           0.8849                0.9785       0.0937
confusable           DoS-UDP_Flood               300           0.1297                0.0000     

In [19]:
import os, subprocess
os.chdir("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
!git config --global user.name "Md Anas Biswas"
!git config --global user.email "anasbiswas@gmail.com"
!git add -A
!git commit -q -m "D4 rarity-vs-separability ablation (the keystone defence): SEPARABILITY causal, rarity held constant. At matched rare count (300 train rows): separable classes mean M0 recall 0.960 (RSTFIN 0.999, ICMP 0.997, Mirai-greip 0.885) vs confusable classes 0.043 (DoS-UDP 0.130, DoS-TCP 0, SynonymousIP 0) - 22x gap. Separable classes also survive prune80 (ICMP 0.991, Mirai 0.979); confusable stay 0. Refutes 'isnt this just rarity' - rarity matched, outcomes opposite. Separability determines learnability AND pruning-survival; rarity only modulates magnitude"
out = subprocess.run(["git","push"], capture_output=True, text=True)
print(out.stdout or "", out.stderr or "")

 To https://github.com/anasbiswas1/iot-trust-compression.git
   1e08886..4b068e6  main -> main



In [20]:
REPO = "/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression"
ADD = r'''

def per_class_calibration(probs, y_true, n_bins: int = 15):
    """Per-class calibration. For samples PREDICTED as a class: ECE, precision, signed
    over-confidence gap. For samples whose TRUE label is the class: mean confidence and
    fraction confidently-wrong (predicted != true at conf>0.5). Confident-wrongness on a
    collapsed class is the decision-layer-collapse signature."""
    import numpy as np, pandas as pd
    conf = probs.max(1); pred = probs.argmax(1)
    classes = np.unique(y_true); rows = {}
    for c in classes:
        pm = pred == c
        if pm.sum() > 0:
            correct_pred = (y_true[pm] == c).astype(int)
            ece_pred = adaptive_ece(conf[pm], correct_pred, n_bins) if pm.sum() >= n_bins else np.nan
            gap_pred = signed_overconfidence_gap(conf[pm], correct_pred)
            precision = float(correct_pred.mean())
        else:
            ece_pred, gap_pred, precision = np.nan, np.nan, np.nan
        tm = y_true == c
        true_conf = float(conf[tm].mean()) if tm.sum() > 0 else np.nan
        cw = float(((pred[tm] != c) & (conf[tm] > 0.5)).mean()) if tm.sum() > 0 else np.nan
        rows[int(c)] = {
            "n_pred_as": int(pm.sum()), "n_true": int(tm.sum()),
            "precision": round(precision, 4) if precision == precision else np.nan,
            "ece_pred_side": round(ece_pred, 4) if ece_pred == ece_pred else np.nan,
            "overconf_gap_pred_side": round(gap_pred, 4) if gap_pred == gap_pred else np.nan,
            "mean_conf_on_true": round(true_conf, 4) if true_conf == true_conf else np.nan,
            "frac_confidently_wrong": round(cw, 4) if cw == cw else np.nan,
        }
    return pd.DataFrame(rows).T


def overall_ece(probs, y_true, n_bins: int = 15):
    import numpy as np
    conf = probs.max(1); correct = (probs.argmax(1) == y_true).astype(int)
    return adaptive_ece(conf, correct, n_bins)
'''
with open(f"{REPO}/src/metrics.py","a") as f:
    f.write(ADD)
print("appended per_class_calibration + overall_ece")

appended per_class_calibration + overall_ece


In [ ]:
import importlib, importlib.util, sys
REPO = '/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression'
from src import data, train, compression, metrics
for m in (data, train, compression, metrics): importlib.reload(m)
import pandas as pd, numpy as np

# ensure CICIoT in memory
if ('df' not in dir()) or ('ransomware' in set(df['label'].unique())):
    df = data.clean(data.load_raw("ciciot2023"), "ciciot2023")
    splits = data.temporal_within_capture_split(df, seed=CFG["anchor_seed"])
print("CICIoT rows:", len(df))

# M0 anchor + the full compression matrix (probs from each cell)
cnn, le, scaler, feat_cols = train.load_anchor("ciciot2023","cnn1d","M0",CFG["anchor_seed"], arch_kwargs={"channels":(64,128)})
mat = compression.build_matrix(cnn, df, "ciciot2023", splits, seed=CFG["anchor_seed"], arch="cnn1d", verbose=False)
tiers = pd.read_csv(PATHS.tables("baseline","cnn1d_M0_tiers.csv"), index_col=0)["final_tier"]

# overall ECE per compression cell + per-class calibration
print("\n" + "="*70 + "\nOVERALL ECE by compression cell:")
percell_overall = {}
percell_calib = {}
for cell, entry in mat.items():
    rec, macro, probs, y_true, y_pred = compression.evaluate_cell(entry, df, splits, le, scaler, feat_cols)
    if probs is None:
        print(f"  {cell}: skipped"); continue
    oce = metrics.overall_ece(probs, y_true)
    percell_overall[cell] = round(oce, 4)
    cal = metrics.per_class_calibration(probs, y_true)
    cal.index = [le.classes_[int(i)] for i in cal.index]
    percell_calib[cell] = cal
    print(f"  {cell:14} overall ECE={oce:.4f}  macroF1={macro:.4f}")

# the hypothesis test: do MEASURABLE classes that collapsed under prune80 show
# confident-wrongness (high frac_confidently_wrong) under prune80 vs M0?
meas = tiers[tiers=="measurable"].index
calM0 = percell_calib["M0"]; calP8 = percell_calib["prune80"]
print("\n" + "="*70 + "\nPER-CLASS CALIBRATION — measurable tier, M0 vs prune80:")
print("(frac_confidently_wrong = model confidently mis-predicts this class's true samples)")
comp = pd.DataFrame({
    "M0_conf_on_true": calM0["mean_conf_on_true"],
    "p80_conf_on_true": calP8["mean_conf_on_true"],
    "M0_frac_conf_wrong": calM0["frac_confidently_wrong"],
    "p80_frac_conf_wrong": calP8["frac_confidently_wrong"],
}).loc[[c for c in meas if c in calM0.index]]
comp["conf_wrong_increase"] = (comp["p80_frac_conf_wrong"] - comp["M0_frac_conf_wrong"]).round(4)
print(comp.round(3).sort_values("conf_wrong_increase", ascending=False).to_string())

print("\nmean confident-wrong increase (measurable tier, M0->prune80):",
      round(comp["conf_wrong_increase"].mean(), 4))

pd.Series(percell_overall).to_csv(PATHS.tables("explain","overall_ece_by_cell.csv"))
comp.to_csv(PATHS.tables("explain","per_class_calibration_prune80.csv"))
calP8.to_csv(PATHS.tables("explain","per_class_calibration_full_prune80.csv"))
print("\nsaved -> results/tables/explain/")

CICIoT rows: 3661696

OVERALL ECE by compression cell:
  M0             overall ECE=0.0336  macroF1=0.5451
  prune50        overall ECE=0.0260  macroF1=0.5242
  prune80        overall ECE=0.1927  macroF1=0.3377


## Neural-collapse geometry (the theory layer)

Per-class ETF deviation + minority-collapse angle at baseline and under each compression level. Rare classes start with least angular margin -> fail first.

In [ ]:
from src import geometry
# etf = geometry.etf_deviation(feats, labels)
# mci = geometry.minority_collapse_index(feats, labels)
# margins = geometry.per_class_margin(logits, labels)

## Tournament + per-family causes + rarity-vs-separability ablation (D4)

Dissociate pruning vs quantisation vs distillation per class; subsample a frequent class to a rare class's count to separate rarity from separability.

In [ ]:
# H1 info-loss(probe) / H2 decision-rule(margin) / H3 optimisation(PTQ vs finetune) / H4 rarity-vs-sep

## Causal-claim checkpoint

The geometry must predict collapse CONSISTENTLY across all three architectures, else withdraw the causal claim to correlational.

In [ ]:
# record cross-architecture consistency of geometry->collapse

## End-of-unit (non-negotiable)

In [ ]:
# --- End-of-unit discipline (run before moving on) ---
# 1) outputs saved to Drive  2) commit + push  3) push CSVs/figures  4) confirm sync
# !cd $REPO && git add -A && git commit -m 'nb05: <meaningful message>' && git push
print('Saved under:', PATHS.tables().parent)
